# Clinical TabReD Lite
## Evaluating Tabular Machine Learning Models under Temporal Distribution Shift in Healthcare Data

**Course:** CSCI 447 Machine Learning: Theory and Practice, Fall 2026  
**Project ID:** P034  
**Students:** STUDENT_NAME_1 (STUDENT_ID_1), STUDENT_NAME_2 (STUDENT_ID_2)  
**Instructor:** Dr. Hari Mohan Rai, Assistant Professor, Department of Computer Science, SCAI, Nazarbayev University

This notebook is **self-contained**: it downloads the public NHANES data, prepares it, trains five models,
evaluates them under three protocols and analyses temporal drift. It needs only internet access and a
standard Python 3.10+ Jupyter installation. All code below is the same code as in the project repository
(copied automatically), and every number shown is computed while the notebook runs.

**Contents:** 1 Introduction · 2 Dataset · 3 Data loading · 4 Preprocessing · 5 Exploratory analysis ·
6 Train/test splitting · 7 Cross-validation · 8 Model training · 9 Model evaluation · 10 Temporal drift ·
11 Results and comparison · 12 Limitations and conclusion

### Setup
The first cell checks that the required packages are installed and installs only the missing ones,
pinned to the versions this notebook was tested with.

In [ ]:
# Check the required packages and install ONLY the missing ones (tested versions).
import importlib.util
import sys

REQUIRED = {'pandas': 'pandas==3.1.1',
 'numpy': 'numpy==2.5.4',
 'sklearn': 'scikit-learn==1.9.1',
 'xgboost': 'xgboost==3.4.1',
 'lightgbm': 'lightgbm==4.7.0',
 'scipy': 'scipy==1.18.1',
 'matplotlib': 'matplotlib==3.11.2',
 'requests': 'requests==2.34.2'}
missing = [req for module, req in REQUIRED.items() if importlib.util.find_spec(module) is None]
if missing:
    to_install = " ".join(missing)
    print("Installing missing packages:", to_install)
    %pip install -q {to_install}
else:
    print("All required packages are already installed.")

from importlib.metadata import version
print("Python", sys.version.split()[0])
for module, requirement in REQUIRED.items():
    package = requirement.split("==")[0]
    print(f"{package:14s} installed {version(package):10s} (tested with {requirement.split('==')[1]})")
# The notebook is self-contained: the project package is NOT needed (expected: False).
print("Project package importable:", importlib.util.find_spec("clinical_tabred") is not None)

In [ ]:
%matplotlib inline
# Imports used by the functions below (collected from the project modules).
import copy
import hashlib
import itertools
import json
import logging
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import platform
import requests
import subprocess
import sys
import time
from datetime import datetime, timezone
from importlib import metadata
from lightgbm import LGBMClassifier
from matplotlib import font_manager
from pathlib import Path
from scipy.stats import ks_2samp
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, confusion_matrix, log_loss, precision_recall_curve, roc_auc_score, roc_curve
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier
from IPython.display import Markdown, display

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
logger = logging.getLogger("clinical_tabred")
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

All settings live in one `CONFIG` dictionary. Paths are relative to the notebook:
raw data is cached in `./nhanes_data/`, and all tables and figures are written to `./notebook_outputs/`.

In [ ]:
# All settings in one place (generated from the project configuration file).
# RUN_MODE = "quick" runs a small smoke test; "full" reproduces the reported results.
RUN_MODE = "full"

CONFIG = {'project': {'course': 'CSCI 447 Machine Learning: Theory and Practice',
             'semester': 'Fall 2026',
             'project_id': 'P034',
             'notebook_name': 'CSCI447P034.ipynb',
             'title': 'Clinical TabReD Lite',
             'full_title': 'Evaluating Tabular Machine Learning Models under Temporal '
                           'Distribution Shift in Healthcare Data',
             'students': [{'name': 'STUDENT_NAME_1', 'id': 'STUDENT_ID_1'},
                          {'name': 'STUDENT_NAME_2', 'id': 'STUDENT_ID_2'}],
             'instructor': 'Dr. Hari Mohan Rai, Assistant Professor, Department of Computer '
                           'Science, SCAI, Nazarbayev University'},
 'seed': 42,
 'n_jobs': 4,
 'paths': {'raw_dir': 'nhanes_data',
           'processed_dir': 'notebook_outputs/processed',
           'results_dir': 'notebook_outputs',
           'figures_dir': 'notebook_outputs/figures'},
 'cycles': [{'name': '2011-2012', 'suffix': 'G', 'start_year': 2011, 'end_year': 2012},
            {'name': '2013-2014', 'suffix': 'H', 'start_year': 2013, 'end_year': 2014},
            {'name': '2015-2016', 'suffix': 'I', 'start_year': 2015, 'end_year': 2016}],
 'download': {'components': ['DEMO', 'BMX', 'DIQ'],
              'primary_url': 'https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/{start_year}/DataFiles/{component}_{suffix}.xpt',
              'fallback_url': 'https://wwwn.cdc.gov/Nchs/Nhanes/{start_year}-{end_year}/{component}_{suffix}.XPT',
              'timeout_s': 60,
              'retries': 3,
              'user_agent': 'Mozilla/5.0 (educational research script; clinical-tabred-lite)'},
 'data': {'min_age': 20,
          'target_column': 'DIQ010',
          'positive_code': 1,
          'negative_code': 2,
          'numeric_features': ['RIDAGEYR', 'INDFMPIR', 'BMXBMI', 'BMXWAIST', 'BMXHT', 'BMXWT'],
          'categorical_features': ['RIAGENDR', 'RIDRETH3', 'DMDEDUC2'],
          'special_missing_codes': {'DMDEDUC2': [7, 9]},
          'metadata_columns': ['SEQN', 'cycle', 'WTINT2YR'],
          'feature_groups': {'demographic': ['RIDAGEYR', 'RIAGENDR', 'RIDRETH3'],
                             'socioeconomic': ['DMDEDUC2', 'INDFMPIR'],
                             'anthropometric': ['BMXBMI', 'BMXWAIST', 'BMXHT', 'BMXWT']},
          'prevalence_bounds': [0.05, 0.25],
          'category_labels': {'RIAGENDR': {1: 'Male', 2: 'Female'},
                              'RIDRETH3': {1: 'Mexican American',
                                           2: 'Other Hispanic',
                                           3: 'Non-Hispanic White',
                                           4: 'Non-Hispanic Black',
                                           6: 'Non-Hispanic Asian',
                                           7: 'Other/multi-racial'},
                              'DMDEDUC2': {1: 'Less than 9th grade',
                                           2: '9-11th grade',
                                           3: 'High school/GED',
                                           4: 'Some college/AA',
                                           5: 'College graduate+'}}},
 'drift': {'reference_cycle': '2011-2012', 'psi_bins': 10},
 'protocols': {'temporal': {'train': '2011-2012', 'val': '2013-2014', 'test': '2015-2016'},
               'in_period': {'holdout_fraction': 0.2},
               'random': {'train': 0.7,
                          'val': 0.15,
                          'test': 0.15,
                          'seeds': [42, 43, 44, 45, 46]}},
 'cv': {'n_splits': 5},
 'bootstrap': {'n_resamples': 1000, 'confidence': 0.95},
 'mlp_training': {'max_epochs': 200, 'patience': 10},
 'models': {'logreg': {'C': [0.01, 0.1, 1.0, 10.0]},
            'random_forest': {'n_estimators': [300],
                              'max_depth': [6, 10, None],
                              'min_samples_leaf': [1, 10]},
            'xgboost': {'n_estimators': [300],
                        'learning_rate': [0.03, 0.1],
                        'max_depth': [3, 5],
                        'subsample': [0.8]},
            'lightgbm': {'n_estimators': [300],
                         'learning_rate': [0.03, 0.1],
                         'num_leaves': [7, 31]},
            'mlp': {'hidden_layer_sizes': [[64], [64, 32]], 'alpha': [0.0001, 0.01]}},
 'quick': {'rows_per_cycle': 800, 'n_resamples': 50, 'random_seeds': [42], 'max_epochs': 15}}

def quick_config(config: dict) -> dict:
    """A cheaper copy of the config for smoke tests: one value per grid, few epochs."""
    quick = json.loads(json.dumps(config))  # deep copy
    quick["models"] = {name: {k: v[:1] for k, v in grid.items()}
                       for name, grid in config["models"].items()}
    quick["bootstrap"]["n_resamples"] = config["quick"]["n_resamples"]
    quick["protocols"]["random"]["seeds"] = config["quick"]["random_seeds"]
    quick["mlp_training"]["max_epochs"] = config["quick"]["max_epochs"]
    return quick


if RUN_MODE == "quick":
    CONFIG = quick_config(CONFIG)

RAW_DIR = Path(CONFIG["paths"]["raw_dir"])
PROCESSED_DIR = Path(CONFIG["paths"]["processed_dir"])
RESULTS_DIR = Path(CONFIG["paths"]["results_dir"])
FIGURES_DIR = Path(CONFIG["paths"]["figures_dir"])
for folder in [RAW_DIR, PROCESSED_DIR, RESULTS_DIR, FIGURES_DIR]:
    folder.mkdir(parents=True, exist_ok=True)
print("Run mode:", RUN_MODE, "| seed:", CONFIG["seed"])

## 1. Project introduction

Machine learning models for healthcare are usually trained on data collected in the past and then used on
patients seen later. If the population, measurement practice or disease prevalence changes over time, a
model can perform worse than a standard random train/test split suggests. The TabReD benchmark
(Rubachev et al., 2024) showed that, for industrial tabular datasets, **time-based splits** can change
which methods look best compared with random splits.

This project is a small, educational version of that idea on public health-survey data. We predict
whether an adult reports having been told by a doctor that they have diabetes, using demographic,
socioeconomic and body-measurement features, and compare:

* **Protocol A (chronological):** train on 2011-2012, validate on 2013-2014, test on 2015-2016.
* **Protocol A2 (in-period reference):** the same model, tested both on a held-out 20% of 2011-2012 and on 2015-2016.
* **Protocol B (random):** all cycles pooled and split randomly 70/15/15 (5 seeds).
* **5-fold stratified cross-validation** on the development data of A and B.

**Research questions**
1. How does performance differ between random and chronological splits?
2. Which models generalise better to a newer survey cohort?
3. Which feature distributions and target prevalences change between cycles?
4. Can these observed changes help explain performance differences?

This is a prototype for learning, **not** a validated clinical prediction tool.

## 2. Dataset description

**Source:** CDC National Health and Nutrition Examination Survey (NHANES), public-use files from
<https://wwwn.cdc.gov/nchs/nhanes/>. Three two-year cycles are used: 2011-2012 (file suffix `G`),
2013-2014 (`H`) and 2015-2016 (`I`). From each cycle we use three components:

| Component | Content | Variables used |
|---|---|---|
| DEMO | Demographics | `RIDAGEYR` age (top-coded at 80), `RIAGENDR` gender, `RIDRETH3` race/Hispanic origin, `DMDEDUC2` education (7 = refused, 9 = don't know → missing), `INDFMPIR` family income-to-poverty ratio (top-coded at 5.0), `WTINT2YR` interview weight (description only) |
| BMX | Body measures | `BMXBMI` BMI, `BMXWAIST` waist circumference, `BMXHT` height, `BMXWT` weight |
| DIQ | Diabetes questionnaire | `DIQ010` target |

**Target:** `DIQ010` "Doctor told you have diabetes": 1 = yes → **1**, 2 = no → **0**; 3 (borderline),
7 (refused), 9 (don't know) and missing are **excluded**. This is a cross-sectional classification of
*self-reported diagnosed* diabetes, not a prediction of future onset.

**Population:** adults aged 20 or older with a body-measurement record and a valid label.

**Never used as features:** the respondent ID `SEQN`, the cycle, the target, other diabetes answers,
glucose/HbA1c/insulin laboratory values (they define diabetes, which would make the task trivial),
and survey weights or design variables. Every variable was checked in the official codebook of all three cycles.

NHANES is a complex multistage survey; we train **unweighted** models and only report weighted prevalence
as a descriptive check.

## 3. Data loading

The CDC website sometimes returns an HTML error page with HTTP status 200 instead of the data file.
Therefore every file is **validated**: it must not start with `<`, it must start with the SAS transport
header, and pandas must be able to read it. Valid files are cached in `./nhanes_data/`, and a manifest
records the URL, HTTP status, size and SHA-256 hash of every file. If a download fails, the error message
lists the exact URLs that were tried.

In [ ]:
XPT_MAGIC = b"HEADER RECORD*******LIBRARY HEADER RECORD"


def build_urls(download_cfg: dict, cycle: dict, component: str) -> list[str]:
    """Return the candidate download URLs (primary first, then legacy fallback)."""
    fields = {**cycle, "component": component}
    return [download_cfg["primary_url"].format(**fields), download_cfg["fallback_url"].format(**fields)]


def validate_xpt(path: Path) -> str:
    """Check that a file is a real XPT file. Returns "ok" or a short error message."""
    if not path.exists() or path.stat().st_size == 0:
        return "missing or empty file"
    head = path.read_bytes()[:80]
    if head.lstrip().startswith(b"<"):
        return "file is an HTML page, not XPT"
    if not head.startswith(XPT_MAGIC):
        return "missing SAS transport header"
    try:
        pd.read_sas(path, format="xport").head()
    except Exception as error:  # noqa: BLE001 - any parser error means "invalid file"
        return f"pandas cannot read file: {error}"
    return "ok"


def sha256_of(path: Path) -> str:
    """Return the SHA-256 hex digest of a file (recorded in the manifest)."""
    return hashlib.sha256(path.read_bytes()).hexdigest()


def fetch_url(url: str, target: Path, download_cfg: dict) -> int:
    """Download one URL to `target` with retries. Returns the final HTTP status code."""
    headers = {"User-Agent": download_cfg["user_agent"]}
    status = -1
    for attempt in range(1, download_cfg["retries"] + 1):
        try:
            response = requests.get(url, headers=headers, timeout=download_cfg["timeout_s"])
            status = response.status_code
            if status == 200:
                target.write_bytes(response.content)
                return status
            logger.warning("HTTP %s for %s (attempt %d)", status, url, attempt)
        except requests.RequestException as error:
            logger.warning("Request failed for %s (attempt %d): %s", url, attempt, error)
        time.sleep(2 * attempt)
    return status


def download_component(
    cycle: dict, component: str, download_cfg: dict, raw_dir: Path, previous: dict
) -> dict:
    """Make sure one valid XPT file is cached in `raw_dir`; return its manifest row.

    `previous` maps file names to rows of an earlier manifest, so a cached file
    keeps the URL and HTTP status from the run that actually downloaded it.
    """
    target = raw_dir / f"{component}_{cycle['suffix']}.xpt"
    row = {"component": component, "cycle": cycle["name"], "file": target.name}
    if validate_xpt(target) == "ok":
        logger.info("Using cached %s", target.name)
        old = previous.get(target.name, {})
        return {**row, "url": old.get("url", "cached (source unknown)"),
                "http_status": old.get("http_status"), "bytes": target.stat().st_size,
                "sha256": sha256_of(target), "validation": "ok", "from_cache": True}
    for url in build_urls(download_cfg, cycle, component):
        status = fetch_url(url, target, download_cfg)
        result = validate_xpt(target) if status == 200 else f"HTTP {status}"
        logger.info("%s -> %s (%s)", url, status, result)
        if result == "ok":
            return {**row, "url": url, "http_status": status, "bytes": target.stat().st_size,
                    "sha256": sha256_of(target), "validation": "ok", "from_cache": False}
        target.unlink(missing_ok=True)  # never keep an invalid file in the cache
    raise RuntimeError(
        f"Could not download a valid {component}_{cycle['suffix']} file. Tried: "
        f"{build_urls(download_cfg, cycle, component)}; last result: {result}. "
        "Check your internet connection or download the file manually into "
        f"{raw_dir}."
    )


def download_all(config: dict, raw_dir: Path) -> pd.DataFrame:
    """Download every component of every cycle and write `manifest.csv` to `raw_dir`."""
    raw_dir = Path(raw_dir)
    raw_dir.mkdir(parents=True, exist_ok=True)
    manifest_path = raw_dir / "manifest.csv"
    previous = {}
    if manifest_path.exists():
        previous = pd.read_csv(manifest_path).set_index("file").to_dict("index")
    rows = []
    for cycle in config["cycles"]:
        for component in config["download"]["components"]:
            rows.append(
                download_component(cycle, component, config["download"], raw_dir, previous)
            )
    manifest = pd.DataFrame(rows)
    manifest.to_csv(manifest_path, index=False)
    return manifest

In [ ]:
manifest = download_all(CONFIG, RAW_DIR)
manifest[["component", "cycle", "url", "http_status", "bytes", "validation"]]

Reading an XPT file: `pandas.read_sas` returns all numbers as floats and NHANES stores some zeros as tiny
values such as 5.4e-79, which we round to exactly 0. We also check that the respondent ID is unique.

In [ ]:
def read_xpt(path: Path) -> pd.DataFrame:
    """Read one XPT file and clean the floating-point quirks of `read_sas`.

    NHANES stores some zeros as tiny numbers such as 5.4e-79, so values that are
    numerically zero are rounded to exactly 0.
    """
    df = pd.read_sas(path, format="xport")
    numeric = df.select_dtypes("number").columns
    df[numeric] = df[numeric].mask(df[numeric].abs() < 1e-10, 0.0)
    if not df["SEQN"].is_unique:
        raise ValueError(f"SEQN is not unique in {path}; the file may be corrupted.")
    df["SEQN"] = df["SEQN"].astype("int64")
    return df

In [ ]:
for path in sorted(RAW_DIR.glob("*.xpt")):
    table = read_xpt(path)
    print(f"{path.name:12s} rows={len(table):6,d} columns={table.shape[1]}")

## 4. Data preprocessing

Steps for each cycle:
1. **Merge** DEMO, BMX and DIQ on `SEQN`, starting from DEMO (left join), so nobody is lost silently.
2. **Build the label** from `DIQ010` (1 → 1, 2 → 0, everything else excluded).
3. **Clean codes:** categorical codes become pandas `Int64`; refused/don't-know codes become missing.
4. **Filter** eligible respondents and record every exclusion in a data-flow (attrition) table.
5. **Sanity check:** if the positive rate of a cycle falls outside 5-25%, stop, because the label mapping is probably wrong.

In [ ]:
def merge_cycle(raw_dir: Path, cycle: dict, data_cfg: dict) -> tuple[pd.DataFrame, dict]:
    """Merge DEMO, BMX and DIQ of one cycle on SEQN (left join starting from DEMO).

    Returns the merged table and a dict with how many adults lack each component.
    """
    suffix = cycle["suffix"]
    demo = read_xpt(Path(raw_dir) / f"DEMO_{suffix}.xpt")
    bmx = read_xpt(Path(raw_dir) / f"BMX_{suffix}.xpt")
    diq = read_xpt(Path(raw_dir) / f"DIQ_{suffix}.xpt")
    demo_cols = ["SEQN", "WTINT2YR", "RIDAGEYR", "RIAGENDR", "RIDRETH3", "DMDEDUC2", "INDFMPIR"]
    bmx_cols = ["SEQN"] + [c for c in data_cfg["numeric_features"] if c.startswith("BMX")]
    merged = demo[demo_cols].merge(bmx[bmx_cols], on="SEQN", how="left", validate="one_to_one")
    merged = merged.merge(diq[["SEQN", data_cfg["target_column"]]], on="SEQN", how="left",
                          validate="one_to_one")
    merged["has_bmx"] = merged["SEQN"].isin(bmx["SEQN"])
    merged["has_diq"] = merged["SEQN"].isin(diq["SEQN"])
    merged.insert(1, "cycle", cycle["name"])
    adults = merged[merged["RIDAGEYR"] >= data_cfg["min_age"]]
    missing = {"adults_without_bmx": int((~adults["has_bmx"]).sum()),
               "adults_without_diq": int((~adults["has_diq"]).sum())}
    return merged, missing


def map_target(codes: pd.Series, positive_code: int, negative_code: int) -> pd.Series:
    """Map DIQ010 answers to a binary label: 1 -> 1, 2 -> 0, anything else -> <NA>.

    Codes 3 (borderline), 7 (refused), 9 (don't know) and missing are excluded later,
    because they are neither a clear "yes" nor a clear "no".
    """
    codes = codes.round().astype("Int64")
    label = pd.Series(pd.NA, index=codes.index, dtype="Int64")
    label[codes == positive_code] = 1
    label[codes == negative_code] = 0
    return label


def clean_special_codes(df: pd.DataFrame, data_cfg: dict) -> pd.DataFrame:
    """Cast categorical codes to Int64 and replace refused/don't-know codes with <NA>."""
    df = df.copy()
    for column in data_cfg["categorical_features"]:
        df[column] = df[column].round().astype("Int64")
    for column, codes in data_cfg["special_missing_codes"].items():
        df[column] = df[column].mask(df[column].isin(codes))
    return df


def flow_row(step: str, cycle: str, rows: int, removed: int, reason: str) -> dict:
    """One row of the attrition (data flow) table."""
    return {"step": step, "cycle": cycle, "rows": rows, "removed": removed, "reason": reason}


def filter_cycle(merged: pd.DataFrame, cycle: str, data_cfg: dict) -> tuple[pd.DataFrame, list]:
    """Apply the eligibility filters to one cycle and log each exclusion step."""
    flow = [flow_row("1_demographics", cycle, len(merged), 0, "all DEMO respondents")]
    steps = [
        ("2_adults", merged["RIDAGEYR"] >= data_cfg["min_age"], "age below 20 years"),
        ("3_has_exam", merged["has_bmx"], "no body-measurement (BMX) record: interview only"),
        ("4_has_diq", merged["has_diq"], "no diabetes questionnaire (DIQ) record"),
        ("5_valid_label", merged["label"].notna(), "DIQ010 is 3/7/9 or missing"),
    ]
    keep = pd.Series(True, index=merged.index)
    for step, condition, reason in steps:
        before = int(keep.sum())
        keep &= condition
        flow.append(flow_row(step, cycle, int(keep.sum()), before - int(keep.sum()), reason))
    return merged[keep].copy(), flow


def check_prevalence(dataset: pd.DataFrame, bounds: list) -> None:
    """Stop if the positive rate of any cycle is implausible (likely a target-mapping bug)."""
    prevalence = dataset.groupby("cycle")["label"].mean()
    logger.info("Positive prevalence per cycle:\n%s", prevalence.round(4).to_string())
    low, high = bounds
    bad = prevalence[(prevalence < low) | (prevalence > high)]
    if not bad.empty:
        raise ValueError(
            f"Prevalence outside [{low}, {high}] for {bad.to_dict()}. "
            "Check the DIQ010 target mapping before training any model."
        )


def build_dataset(config: dict, raw_dir: Path) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Build the analysis table for all cycles.

    Returns (dataset, data_flow, missing_components) where data_flow lists every
    exclusion step and missing_components counts adults lacking BMX or DIQ records.
    """
    data_cfg = config["data"]
    tables, flow, missing_rows = [], [], []
    for cycle in config["cycles"]:
        merged, missing = merge_cycle(raw_dir, cycle, data_cfg)
        merged["label"] = map_target(merged[data_cfg["target_column"]],
                                     data_cfg["positive_code"], data_cfg["negative_code"])
        merged = clean_special_codes(merged, data_cfg)
        kept, cycle_flow = filter_cycle(merged, cycle["name"], data_cfg)
        tables.append(kept)
        flow.extend(cycle_flow)
        missing_rows.append({"cycle": cycle["name"], **missing})
    features = data_cfg["numeric_features"] + data_cfg["categorical_features"]
    dataset = pd.concat(tables, ignore_index=True)[["SEQN", "cycle", "WTINT2YR"] + features
                                                    + ["label"]]
    check_prevalence(dataset, data_cfg["prevalence_bounds"])
    return dataset, pd.DataFrame(flow), pd.DataFrame(missing_rows)


def missingness_table(dataset: pd.DataFrame, features: list[str]) -> pd.DataFrame:
    """Percentage of missing values per feature and cycle (rows: features)."""
    return (dataset.groupby("cycle")[features].agg(lambda s: s.isna().mean() * 100).T
            .rename_axis("feature").reset_index())


def save_processed(dataset: pd.DataFrame, config: dict, processed_dir: Path) -> None:
    """Save the processed table as CSV plus a JSON schema describing every column."""
    processed_dir = Path(processed_dir)
    processed_dir.mkdir(parents=True, exist_ok=True)
    dataset.to_csv(processed_dir / "nhanes_adults.csv", index=False)
    data_cfg = config["data"]
    roles = {c: "numeric_feature" for c in data_cfg["numeric_features"]}
    roles.update({c: "categorical_feature" for c in data_cfg["categorical_features"]})
    roles.update({"SEQN": "id", "cycle": "metadata", "WTINT2YR": "survey_weight_metadata",
                  "label": "target"})
    schema = {c: {"role": roles[c], "dtype": str(dataset[c].dtype)} for c in dataset.columns}
    for c in data_cfg["categorical_features"]:
        schema[c]["categories"] = sorted(int(v) for v in dataset[c].dropna().unique())
    (processed_dir / "schema.json").write_text(json.dumps(schema, indent=2))


def subsample_per_cycle(dataset: pd.DataFrame, n_rows: int, seed: int) -> pd.DataFrame:
    """Keep at most `n_rows` random rows per cycle (used by the --quick smoke test)."""
    parts = [g.sample(n=min(n_rows, len(g)), random_state=seed)
             for _, g in dataset.groupby("cycle")]
    return pd.concat(parts).sort_index().reset_index(drop=True)


def feature_matrix(dataset: pd.DataFrame, features: list[str]) -> pd.DataFrame:
    """Return the model inputs as floats (pd.NA -> np.nan) so sklearn can impute them."""
    return dataset[features].astype("float64").replace({pd.NA: np.nan})

In [ ]:
dataset, data_flow, missing_components = build_dataset(CONFIG, RAW_DIR)
save_processed(dataset, CONFIG, PROCESSED_DIR)
FEATURES = CONFIG["data"]["numeric_features"] + CONFIG["data"]["categorical_features"]
data_flow.to_csv(RESULTS_DIR / "data_flow.csv", index=False)
missing_components.to_csv(RESULTS_DIR / "missing_components.csv", index=False)
missingness_table(dataset, FEATURES).to_csv(RESULTS_DIR / "missingness.csv", index=False)
if RUN_MODE == "quick":
    dataset = subsample_per_cycle(dataset, CONFIG["quick"]["rows_per_cycle"], CONFIG["seed"])
print("Final dataset:", dataset.shape)
display(data_flow)
display(missing_components)
dataset.head()

Missing values per feature and cycle (in %). They are imputed inside the model pipeline (below), never here:

In [ ]:
missingness_table(dataset, FEATURES).round(2)

### The preprocessing pipeline
The same `ColumnTransformer` is used by all five models:
numeric features → median imputation → standardisation; categorical codes → most-frequent imputation →
one-hot encoding. It is **fitted on the training split only** and then applied unchanged to validation
and test data, so no statistics from the evaluation data leak into training.

In [ ]:
def make_preprocessor(numeric: list[str], categorical: list[str]) -> ColumnTransformer:
    """Imputation + scaling for numbers, imputation + one-hot encoding for categories.

    The same preprocessor is used for every model. It is always fitted on the
    training split only: if it saw validation or test rows, their medians and
    category frequencies would leak information about data the model must not see.
    """
    numeric_steps = Pipeline([("impute", SimpleImputer(strategy="median")),
                              ("scale", StandardScaler())])
    categorical_steps = Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        # Category codes (e.g. race 1..7) are labels, not quantities, so they are one-hot encoded.
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])
    return ColumnTransformer([("num", numeric_steps, numeric),
                              ("cat", categorical_steps, categorical)])

In [ ]:
make_preprocessor(CONFIG["data"]["numeric_features"], CONFIG["data"]["categorical_features"])

## 5. Exploratory data analysis

We first summarise each cycle: sample size, unweighted and survey-weighted prevalence, and a few means.
The figure functions read the saved CSV files, so every plot shows the stored results.

In [ ]:
def weighted_mean(values: pd.Series, weights: pd.Series) -> float:
    """Survey-weighted mean (ignores rows where the value or weight is missing)."""
    ok = values.notna() & weights.notna() & (weights > 0)
    return float(np.average(values[ok].astype(float), weights=weights[ok]))


def cycle_summary(dataset: pd.DataFrame) -> pd.DataFrame:
    """Sample size, prevalence (unweighted and survey-weighted) and key means per cycle.

    The weighted prevalence uses the interview weight WTINT2YR (the diabetes
    question is part of the interview). It is a descriptive check only; all
    models are trained without weights.
    """
    rows = []
    for cycle, part in dataset.groupby("cycle"):
        rows.append({
            "cycle": cycle, "n": len(part), "n_pos": int(part["label"].sum()),
            "prevalence": part["label"].mean(),
            "weighted_prevalence": weighted_mean(part["label"], part["WTINT2YR"]),
            "mean_age": part["RIDAGEYR"].mean(), "mean_bmi": part["BMXBMI"].mean(),
            "mean_waist": part["BMXWAIST"].mean(), "female_share": (part["RIAGENDR"] == 2).mean(),
            "any_missing_share": part.drop(columns=["WTINT2YR"]).isna().any(axis=1).mean(),
        })
    return pd.DataFrame(rows)

In [ ]:
summary = cycle_summary(dataset)
summary.to_csv(RESULTS_DIR / "cycle_summary.csv", index=False)
display(summary.round(4))
dataset[FEATURES].describe().T.round(2)

Plot style and helpers (colours follow the course presentation theme; model colours were checked for
colour-blind separation):

In [ ]:
MODEL_LABELS = {"logreg": "Logistic Regression", "random_forest": "Random Forest",
                "xgboost": "XGBoost", "lightgbm": "LightGBM", "mlp": "MLP"}


MODEL_COLORS = {"logreg": "#1F5FA8", "random_forest": "#A8822C", "xgboost": "#109A92",
                "lightgbm": "#B4432E", "mlp": "#6B4E9B"}


CYCLE_COLORS = ["#9FB0CC", "#4F6A99", "#14213D"]


INK, MUTED, GRID = "#1F2937", "#5B6472", "#D5DAE3"


def apply_style(style: str) -> None:
    """Set matplotlib defaults. 'paper' for the report/notebook, 'slide' for 16:9 slides."""
    fonts = [f for f in ["Calibri", "Carlito", "Arial", "DejaVu Sans"]
             if f in {entry.name for entry in font_manager.fontManager.ttflist}]
    base = 15 if style == "slide" else 10
    plt.rcParams.update({
        "font.family": fonts[0], "font.size": base, "axes.titlesize": base + 2,
        "axes.labelsize": base, "xtick.labelsize": base - 1, "ytick.labelsize": base - 1,
        "legend.fontsize": base - 1, "axes.edgecolor": MUTED, "axes.labelcolor": INK,
        "xtick.color": MUTED, "ytick.color": MUTED, "text.color": INK,
        "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
        "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True,
        "figure.facecolor": "white", "savefig.facecolor": "white", "legend.frameon": False,
    })


def figure_size(style: str, height_ratio: float = 0.45) -> tuple[float, float]:
    """Width x height in inches: wide 16:9-friendly for slides, column width for paper."""
    width = 12.0 if style == "slide" else 7.0
    return width, width * height_ratio


def bottom_legend(fig, ax, ncol: int, height: float = 0.14) -> None:
    """Place the legend of `ax` below the plot so it never covers data."""
    handles, labels = ax.get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=ncol)
    fig.tight_layout(rect=(0, height, 1, 1))


def models_in(table: pd.DataFrame) -> list[str]:
    """Model names in the fixed display order (so colours never change between plots)."""
    return [m for m in MODEL_LABELS if m in set(table["model"])]


def save_figure(fig, out_dir: Path, name: str, formats: list[str]) -> list[Path]:
    """Save one figure as e.g. PNG (300 DPI) and PDF; returns the written paths."""
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    paths = [Path(out_dir) / f"{name}.{fmt}" for fmt in formats]
    for path in paths:
        fig.savefig(path, dpi=300, bbox_inches="tight")
    return paths

In [ ]:
def plot_dataset_characteristics(results_dir: Path, style: str):
    """Fig. 1: respondents per cycle and the share of missing values per feature."""
    summary = pd.read_csv(Path(results_dir) / "cycle_summary.csv")
    missing = pd.read_csv(Path(results_dir) / "missingness.csv").set_index("feature")
    missing = missing[(missing > 0).any(axis=1)]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=figure_size(style),
                                   gridspec_kw={"width_ratios": [1, 2]})
    bars = ax1.bar(summary["cycle"], summary["n"], color=CYCLE_COLORS, width=0.6)
    ax1.bar_label(bars, fmt="{:,.0f}", padding=2)
    ax1.set(title="Adults with a valid label", ylabel="Respondents")
    ax1.set_ylim(0, summary["n"].max() * 1.15)
    width = 0.8 / len(missing.columns)
    for i, cycle in enumerate(missing.columns):
        ax2.bar(np.arange(len(missing)) + i * width, missing[cycle], width * 0.92,
                color=CYCLE_COLORS[i], label=cycle)
    ax2.set_xticks(np.arange(len(missing)) + width, missing.index, rotation=30, ha="right")
    ax2.set(title="Missing values per feature", ylabel="Missing (%)")
    ax2.legend(title="Cycle")
    fig.tight_layout()
    return fig


def plot_prevalence(results_dir: Path, style: str):
    """Fig. 2: share of adults reporting diagnosed diabetes, unweighted and survey-weighted."""
    summary = pd.read_csv(Path(results_dir) / "cycle_summary.csv")
    fig, ax = plt.subplots(figsize=figure_size(style, 0.5 if style == "paper" else 0.42))
    x = np.arange(len(summary))
    for offset, column, color, label in [(-0.18, "prevalence", "#14213D", "Unweighted (sample)"),
                                         (0.18, "weighted_prevalence", "#A8822C",
                                          "Survey-weighted (WTINT2YR)")]:
        bars = ax.bar(x + offset, summary[column] * 100, 0.34, color=color, label=label)
        ax.bar_label(bars, fmt="{:.1f}%", padding=2)
    ax.set_xticks(x, summary["cycle"])
    ax.set(ylabel="Self-reported diagnosed diabetes (%)", title="Target prevalence by cycle")
    ax.set_ylim(0, summary["prevalence"].max() * 100 * 1.25)
    ax.legend(loc="upper left")
    fig.tight_layout()
    return fig

In [ ]:
apply_style("paper")
for name, plot in [("fig01_dataset_characteristics", plot_dataset_characteristics),
                   ("fig02_prevalence", plot_prevalence)]:
    fig = plot(RESULTS_DIR, "paper")
    save_figure(fig, FIGURES_DIR, name, ["png", "pdf"])
    plt.show()

Category shares by cycle (gender, race/Hispanic origin, education):

In [ ]:
for column in CONFIG["data"]["categorical_features"]:
    shares = pd.crosstab(dataset[column].astype("Float64").fillna(-1), dataset["cycle"],
                         normalize="columns") * 100
    shares.index = [CONFIG["data"]["category_labels"][column].get(int(c), "missing")
                    for c in shares.index]
    display(shares.round(1).rename_axis(column))

## 6. Train/test splitting

| Protocol | Train | Validation | Test |
|---|---|---|---|
| A chronological | 2011-2012 | 2013-2014 | 2015-2016 |
| A2 in-period reference | 80% of 2011-2012 (stratified) | 2013-2014 | 20% of 2011-2012 **and** 2015-2016 |
| B random | 70% of all cycles | 15% | 15% (stratified, 5 seeds) |

Validation data is used for hyperparameter selection, the MLP's early stopping and the decision threshold.
Test data is used only once, for the final evaluation. We check that no respondent appears in two splits.
Note that random and chronological test sets contain different people, so a difference between A and B
cannot be attributed entirely to time; protocol A2 isolates the time effect better because it scores the
**same fitted model** on two test sets.

In [ ]:
def temporal_split(dataset: pd.DataFrame, temporal_cfg: dict) -> dict[str, pd.DataFrame]:
    """Split by survey cycle so that the model is always tested on a later period."""
    return {
        "train": dataset[dataset["cycle"] == temporal_cfg["train"]],
        "val": dataset[dataset["cycle"] == temporal_cfg["val"]],
        "test": dataset[dataset["cycle"] == temporal_cfg["test"]],
    }


def in_period_split(
    dataset: pd.DataFrame, temporal_cfg: dict, holdout_fraction: float, seed: int
) -> dict[str, pd.DataFrame]:
    """Protocol A2: one model, evaluated on an in-period holdout and on the future cycle.

    The training cycle is split 80/20 (stratified by label). Validation is still the
    2013-2014 cycle, exactly as in protocol A, so the only thing that differs between
    the two test sets is the time period the respondents come from.
    """
    first = dataset[dataset["cycle"] == temporal_cfg["train"]]
    train, holdout = train_test_split(
        first, test_size=holdout_fraction, stratify=first["label"], random_state=seed
    )
    return {
        "train": train,
        "val": dataset[dataset["cycle"] == temporal_cfg["val"]],
        "test_in_period": holdout,
        "test": dataset[dataset["cycle"] == temporal_cfg["test"]],
    }


def random_split(dataset: pd.DataFrame, random_cfg: dict, seed: int) -> dict[str, pd.DataFrame]:
    """Protocol B: pool all cycles and make a stratified 70/15/15 split.

    Here the test set contains people from every period, so it measures
    performance without any time gap between training and testing.
    """
    test_size = random_cfg["test"]
    train_val, test = train_test_split(
        dataset, test_size=test_size, stratify=dataset["label"], random_state=seed
    )
    val_share = random_cfg["val"] / (1.0 - test_size)  # fraction of the remaining rows
    train, val = train_test_split(
        train_val, test_size=val_share, stratify=train_val["label"], random_state=seed
    )
    return {"train": train, "val": val, "test": test}


def check_no_overlap(splits: dict[str, pd.DataFrame]) -> None:
    """Raise an error if any respondent (SEQN) appears in more than one split."""
    names = list(splits)
    for i, first in enumerate(names):
        for second in names[i + 1:]:
            shared = set(splits[first]["SEQN"]) & set(splits[second]["SEQN"])
            if shared:
                raise ValueError(f"{len(shared)} respondents appear in both {first} and {second}")


def describe_splits(splits: dict[str, pd.DataFrame], protocol: str) -> pd.DataFrame:
    """Size, number of positives and prevalence of every split (for reporting)."""
    rows = []
    for name, part in splits.items():
        rows.append({"protocol": protocol, "split": name, "n": len(part),
                     "n_pos": int(part["label"].sum()), "prevalence": part["label"].mean(),
                     "cycles": ",".join(sorted(part["cycle"].unique()))})
    return pd.DataFrame(rows)


def make_all_splits(dataset: pd.DataFrame, config: dict) -> list[tuple[str, int, dict]]:
    """Build (protocol, seed, splits) for A, A2 and every seed of B; check for overlap."""
    protocols, seed = config["protocols"], config["seed"]
    runs = [("temporal", seed, temporal_split(dataset, protocols["temporal"])),
            ("in_period", seed, in_period_split(dataset, protocols["temporal"],
                                                protocols["in_period"]["holdout_fraction"], seed))]
    runs += [("random", s, random_split(dataset, protocols["random"], s))
             for s in protocols["random"]["seeds"]]
    for _, _, splits in runs:
        check_no_overlap(splits)
    return runs

In [ ]:
all_splits = make_all_splits(dataset, CONFIG)
split_table = pd.concat([describe_splits(s, f"{p}_seed{seed}") for p, seed, s in all_splits])
split_table.round(4)

## 7. Cross-validation

**Strategy:** 5-fold *stratified* cross-validation (each fold keeps the ~13% positive rate) on the
development data only: the 2011-2012 training cycle for protocol A, and train + validation for protocol B.
Test data is never part of cross-validation. Each model uses the hyperparameters selected on validation;
the MLP trains for the number of epochs at which early stopping halted. Threshold metrics use 0.5.

**How CV differs from chronological evaluation:** CV folds are random subsets of the *same* period, so they
estimate in-distribution (i.i.d.) performance and its variability (mean ± std over folds). The
chronological test asks a different question: how well does a model trained in the past work on a
*later* cohort? Agreement between the two suggests little temporal degradation; a gap suggests drift.

The metric functions are defined here because cross-validation and the main evaluation both use them.
Undefined metrics (e.g. precision with no positive predictions) are stored as NaN with a warning,
never as 0.

In [ ]:
def safe_ratio(numerator: float, denominator: float, name: str) -> float:
    """Divide, but return NaN (and log a warning) instead of crashing or writing 0."""
    if denominator == 0:
        logger.warning("%s is undefined (division by zero); stored as NaN", name)
        return float("nan")
    return numerator / denominator


def ranking_metrics(y: np.ndarray, prob: np.ndarray) -> dict:
    """ROC-AUC and PR-AUC (average precision). Both need both classes to be present."""
    if len(np.unique(y)) < 2:
        logger.warning("Only one class present: ROC-AUC and PR-AUC stored as NaN")
        return {"roc_auc": float("nan"), "pr_auc": float("nan")}
    return {"roc_auc": roc_auc_score(y, prob), "pr_auc": average_precision_score(y, prob)}


def compute_metrics(y, prob, threshold: float) -> dict:
    """All metrics for one set of predictions.

    Probabilities >= threshold are called positive. Returns accuracy, precision,
    recall, F1, ROC-AUC, PR-AUC, Brier score and the confusion-matrix counts.
    """
    y, prob = np.asarray(y).astype(int), np.asarray(prob, dtype=float)
    pred = (prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    precision = safe_ratio(tp, tp + fp, "precision")
    recall = safe_ratio(tp, tp + fn, "recall")
    f1 = safe_ratio(2 * tp, 2 * tp + fp + fn, "f1")
    return {"accuracy": (tp + tn) / len(y), "precision": precision, "recall": recall,
            "f1": f1, **ranking_metrics(y, prob), "brier": brier_score_loss(y, prob),
            "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
            "threshold": threshold, "n": len(y), "n_pos": int(y.sum())}


def best_f1_threshold(y_val, prob_val) -> float:
    """Return the probability threshold that maximises F1 on the validation set.

    The threshold is chosen on validation data only; using the test set here would
    make the reported test scores optimistic.
    """
    precision, recall, thresholds = precision_recall_curve(y_val, prob_val)
    f1 = 2 * precision * recall / np.clip(precision + recall, 1e-12, None)
    return float(thresholds[np.argmax(f1[:-1])])  # the last P/R pair has no threshold


def cross_validate(protocol: str, dev: pd.DataFrame, params: dict, best_epoch: dict,
                   config: dict, settings: dict) -> list[dict]:
    """Stratified K-fold CV on development data only (the test set is never touched).

    Each model uses the hyperparameters selected for this protocol. The MLP has no
    validation set inside a fold, so it trains for the number of epochs at which
    early stopping halted in the main run. Threshold-based metrics use 0.5.
    """
    folds = StratifiedKFold(n_splits=config["cv"]["n_splits"], shuffle=True,
                            random_state=config["seed"])
    features = settings["numeric"] + settings["categorical"]
    X, y = feature_matrix(dev, features), dev["label"].astype(int)
    rows = []
    for name in MODEL_NAMES:
        fold_settings = {**settings, "max_epochs": best_epoch[name] or settings["max_epochs"]}
        for fold, (train_idx, test_idx) in enumerate(folds.split(X, y), start=1):
            fitted = fit_model(name, params[name], X.iloc[train_idx], y.iloc[train_idx],
                               None, None, fold_settings)
            prob, _ = predict_proba(fitted["pipeline"], X.iloc[test_idx])
            metrics = compute_metrics(y.iloc[test_idx], prob, 0.5)
            rows.append({"protocol": protocol, "model": name, "fold": fold,
                         **{k: metrics[k] for k in ["accuracy", "precision", "recall", "f1",
                                                    "roc_auc", "pr_auc"]}})
    return rows

Cross-validation is executed at the end of Section 8, because it needs the hyperparameters that are
selected there.

## 8. Model training

Five models share the same preprocessing pipeline:
1. **Logistic Regression** (`class_weight="balanced"`)
2. **Random Forest** (`class_weight="balanced_subsample"`)
3. **XGBoost** (`tree_method="hist"`, `scale_pos_weight = n_neg / n_pos` from the training data)
4. **LightGBM** (`scale_pos_weight`, deterministic mode)
5. **MLP** (scikit-learn `MLPClassifier`), trained with an explicit epoch loop

Only about 13% of respondents are positive, so every model uses class-imbalance handling. scikit-learn's
MLP has no `class_weight`, so it receives "balanced" **sample weights** instead.

Each model has a small grid of at most 6 configurations. Every configuration is trained on the training
split and scored on validation; the one with the best validation **PR-AUC** (ROC-AUC breaks ties) is kept.
There is no refit on train + validation, which keeps the temporal setup clean.

In [ ]:
MODEL_NAMES = ["logreg", "random_forest", "xgboost", "lightgbm", "mlp"]


def expand_grid(grid: dict) -> list[dict]:
    """Turn {"a": [1, 2], "b": [3]} into [{"a": 1, "b": 3}, {"a": 2, "b": 3}]."""
    keys = list(grid)
    return [dict(zip(keys, values)) for values in itertools.product(*grid.values())]


def make_estimator(name: str, params: dict, y_train: pd.Series, seed: int, n_jobs: int,
                   balanced: bool = True):
    """Create one (unfitted) classifier.

    `balanced=True` turns on class-imbalance handling: only ~13% of respondents are
    positive, so without it the models would mostly learn to predict "no diabetes".
    """
    ratio = float((y_train == 0).sum() / (y_train == 1).sum()) if balanced else 1.0
    if name == "logreg":
        return LogisticRegression(C=params["C"], max_iter=2000,
                                  class_weight="balanced" if balanced else None)
    if name == "random_forest":
        return RandomForestClassifier(
            **params, random_state=seed, n_jobs=n_jobs,
            class_weight="balanced_subsample" if balanced else None)
    if name == "xgboost":
        return XGBClassifier(**params, tree_method="hist", scale_pos_weight=ratio,
                             eval_metric="logloss", random_state=seed, n_jobs=n_jobs)
    if name == "lightgbm":
        return LGBMClassifier(**params, scale_pos_weight=ratio, random_state=seed,
                              n_jobs=n_jobs, deterministic=True, force_row_wise=True,
                              verbose=-1)
    if name == "mlp":
        return MLPClassifier(hidden_layer_sizes=tuple(params["hidden_layer_sizes"]),
                             alpha=params["alpha"], learning_rate_init=1e-3, batch_size=128,
                             random_state=seed)
    raise ValueError(f"Unknown model name '{name}'. Choose one of {MODEL_NAMES}.")

### MLP training loop
`partial_fit` performs one pass (epoch) over the training data. After each epoch we record train and
validation log loss and accuracy. Training stops when validation loss has not improved for `patience`
epochs, and the weights of the best epoch are kept. These recorded curves are therefore the real
training curves of the evaluated MLP.

In [ ]:
def epoch_scores(model, X, y, weights) -> tuple[float, float]:
    """Log loss and accuracy (threshold 0.5) of the current MLP on one data split."""
    prob = model.predict_proba(X)[:, 1]
    loss = log_loss(y, prob, sample_weight=weights, labels=[0, 1])
    accuracy = float(np.mean((prob >= 0.5) == y))
    return loss, accuracy


def train_mlp(model: MLPClassifier, X_train, y_train, X_val, y_val, max_epochs: int,
              patience: int, balanced: bool) -> tuple[MLPClassifier, pd.DataFrame, int]:
    """Train the MLP one epoch at a time with early stopping on validation loss.

    Each `partial_fit` call is one pass (epoch) over the training data. sklearn's
    MLP has no `class_weight`, so imbalance is handled with "balanced" sample
    weights; the same weighting is used for the losses we record. If no validation
    data is given, the loop simply runs `max_epochs` epochs.

    Returns (model with the best epoch's weights, per-epoch history, best epoch).
    """
    y_train, y_val = np.asarray(y_train), (None if y_val is None else np.asarray(y_val))
    w_train = compute_sample_weight("balanced", y_train) if balanced else None
    w_val = compute_sample_weight("balanced", y_val) if balanced and y_val is not None else None
    best_model, best_loss, best_epoch, history = model, np.inf, 0, []
    for epoch in range(1, max_epochs + 1):
        model.partial_fit(X_train, y_train, classes=[0, 1], sample_weight=w_train)
        train_loss, train_acc = epoch_scores(model, X_train, y_train, w_train)
        row = {"epoch": epoch, "train_loss": train_loss, "train_accuracy": train_acc}
        if X_val is None:
            history.append(row)
            best_model, best_epoch = model, epoch
            continue
        row["val_loss"], row["val_accuracy"] = epoch_scores(model, X_val, y_val, w_val)
        history.append(row)
        if row["val_loss"] < best_loss - 1e-4:  # a real improvement: remember these weights
            best_loss, best_epoch, best_model = row["val_loss"], epoch, copy.deepcopy(model)
        elif epoch - best_epoch >= patience:  # no improvement for `patience` epochs: stop
            break
    history = pd.DataFrame(history)
    history["best_epoch"] = best_epoch
    return best_model, history, best_epoch


def fit_model(name: str, params: dict, X_train: pd.DataFrame, y_train: pd.Series,
              X_val, y_val, settings: dict) -> dict:
    """Fit preprocessing + model on the training split and return a dict with the fitted
    pipeline, the training time in seconds and (MLP only) the training history.

    `settings` holds numeric/categorical feature lists, seed, n_jobs, balanced,
    max_epochs and patience.
    """
    start = time.perf_counter()
    preprocessor = make_preprocessor(settings["numeric"], settings["categorical"])
    estimator = make_estimator(name, params, y_train, settings["seed"], settings["n_jobs"],
                               settings["balanced"])
    history, best_epoch = None, None
    if name == "mlp":
        Xt_train = preprocessor.fit_transform(X_train)
        Xt_val = None if X_val is None else preprocessor.transform(X_val)
        estimator, history, best_epoch = train_mlp(
            estimator, Xt_train, y_train, Xt_val, y_val, settings["max_epochs"],
            settings["patience"], settings["balanced"])
        pipeline = Pipeline([("prep", preprocessor), ("model", estimator)])
    else:
        pipeline = Pipeline([("prep", preprocessor), ("model", estimator)])
        pipeline.fit(X_train, y_train)
    return {"pipeline": pipeline, "train_time_s": time.perf_counter() - start,
            "history": history, "best_epoch": best_epoch}


def predict_proba(pipeline: Pipeline, X: pd.DataFrame) -> tuple[np.ndarray, float]:
    """Positive-class probabilities and inference time in milliseconds per 1,000 rows."""
    start = time.perf_counter()
    prob = pipeline.predict_proba(X)[:, 1]
    ms_per_1k = (time.perf_counter() - start) * 1000 / max(len(X), 1) * 1000
    return prob, ms_per_1k

### Tuning and evaluation for one protocol

In [ ]:
def base_settings(config: dict) -> dict:
    """Settings shared by every model fit (features, seed, n_jobs, MLP loop length)."""
    data_cfg = config["data"]
    return {"numeric": list(data_cfg["numeric_features"]),
            "categorical": list(data_cfg["categorical_features"]),
            "seed": config["seed"], "n_jobs": config["n_jobs"], "balanced": True,
            "max_epochs": config["mlp_training"]["max_epochs"],
            "patience": config["mlp_training"]["patience"]}


def fit_on_splits(name: str, params: dict, splits: dict, settings: dict) -> dict:
    """Fit one model on splits["train"], using splits["val"] for MLP early stopping."""
    features = settings["numeric"] + settings["categorical"]
    return fit_model(name, params, feature_matrix(splits["train"], features),
                     splits["train"]["label"].astype(int),
                     feature_matrix(splits["val"], features), splits["val"]["label"].astype(int),
                     settings)


def validation_scores(fitted: dict, val: pd.DataFrame, settings: dict) -> tuple:
    """Validation PR-AUC and ROC-AUC of a fitted model (used for model selection)."""
    prob, _ = predict_proba(fitted["pipeline"],
                            feature_matrix(val, settings["numeric"] + settings["categorical"]))
    y = val["label"].astype(int)
    return average_precision_score(y, prob), roc_auc_score(y, prob)


def tune_model(name: str, grid: dict, splits: dict, settings: dict, tag: dict):
    """Try every configuration in the grid and keep the best by validation PR-AUC.

    ROC-AUC breaks ties. Returns (best fitted model, best params, one row per config).
    """
    rows, best = [], None
    for params in expand_grid(grid):
        fitted = fit_on_splits(name, params, splits, settings)
        pr_auc, roc_auc = validation_scores(fitted, splits["val"], settings)
        rows.append({**tag, "model": name, "params": json.dumps(params), "val_pr_auc": pr_auc,
                     "val_roc_auc": roc_auc, "train_time_s": fitted["train_time_s"]})
        if best is None or (pr_auc, roc_auc) > best[0]:
            best = ((pr_auc, roc_auc), fitted, params)
    for row in rows:
        row["selected"] = row["params"] == json.dumps(best[2])
    return best[1], best[2], rows


def evaluate_model(name: str, fitted: dict, splits: dict, settings: dict, tag: dict):
    """Score a fitted model on every split at threshold 0.5 and at the tuned threshold.

    The tuned threshold maximises F1 on the validation split. Returns
    (metric rows, prediction table for the test splits, chosen threshold).
    """
    features = settings["numeric"] + settings["categorical"]
    probs, timings = {}, {}
    for split_name, part in splits.items():
        probs[split_name], timings[split_name] = predict_proba(
            fitted["pipeline"], feature_matrix(part, features))
    threshold = best_f1_threshold(splits["val"]["label"].astype(int), probs["val"])
    rows, predictions = [], []
    for split_name, part in splits.items():
        for rule, value in [("0.5", 0.5), ("val_f1", threshold)]:
            metrics = compute_metrics(part["label"].astype(int), probs[split_name], value)
            rows.append({**tag, "model": name, "split": split_name, "threshold_rule": rule,
                         **metrics, "train_time_s": fitted["train_time_s"],
                         "infer_ms_per_1k": timings[split_name]})
        if split_name.startswith("test"):
            predictions.append(pd.DataFrame({
                **tag, "model": name, "split": split_name, "SEQN": part["SEQN"].to_numpy(),
                "cycle": part["cycle"].to_numpy(), "label": part["label"].astype(int).to_numpy(),
                "probability": probs[split_name]}))
    return rows, pd.concat(predictions, ignore_index=True), threshold


def run_protocol(protocol: str, seed: int, splits: dict, config: dict, settings: dict,
                 fixed_params: dict | None = None) -> dict:
    """Tune (unless `fixed_params` is given), fit and evaluate all five models.

    Returns a dict of lists: metrics, tuning, predictions (per model), history,
    plus the selected params and best MLP epoch for each model.
    """
    tag = {"protocol": protocol, "seed": seed}
    out = {"metrics": [], "tuning": [], "predictions": {}, "history": [], "params": {},
           "best_epoch": {}, "thresholds": {}}
    for name in MODEL_NAMES:
        if fixed_params is None:
            fitted, params, rows = tune_model(name, config["models"][name], splits, settings, tag)
            out["tuning"].extend(rows)
        else:
            params = fixed_params[name]
            fitted = fit_on_splits(name, params, splits, settings)
        metrics, predictions, threshold = evaluate_model(name, fitted, splits, settings, tag)
        out["metrics"].extend(metrics)
        out["predictions"][name] = predictions
        out["params"][name], out["thresholds"][name] = params, threshold
        out["best_epoch"][name] = fitted["best_epoch"]
        if fitted["history"] is not None:
            out["history"].append(fitted["history"].assign(**tag))
        logger.info("%s seed=%s %s params=%s threshold=%.3f", protocol, seed, name, params,
                    threshold)
    return out

In [ ]:
def run_all_protocols(dataset: pd.DataFrame, config: dict) -> dict:
    """Run protocol A (tuned), A2 (A's hyperparameters) and B (tuned per seed).

    Returns {(protocol, seed): run output} plus the split description table.
    """
    settings = base_settings(config)
    runs = make_all_splits(dataset, config)
    outputs = {}
    split_info = pd.concat([describe_splits(s, f"{p}_seed{seed}") for p, seed, s in runs])
    for protocol, seed, splits in runs:
        fixed = outputs[("temporal", config["seed"])]["params"] if protocol == "in_period" else None
        outputs[(protocol, seed)] = run_protocol(protocol, seed, splits, config,
                                                 {**settings, "seed": seed}, fixed)
        outputs[(protocol, seed)]["splits"] = splits
    return {"outputs": outputs, "split_info": split_info}


def save_protocol_results(runs: dict, results_dir: Path) -> dict[str, pd.DataFrame]:
    """Write metrics, tuning, predictions, MLP history and confusion matrices to CSV."""
    results_dir = Path(results_dir)
    (results_dir / "predictions").mkdir(parents=True, exist_ok=True)
    outputs = runs["outputs"].values()
    metrics = pd.DataFrame([row for out in outputs for row in out["metrics"]])
    tables = {
        "metrics": metrics,
        "tuning": pd.DataFrame([row for out in outputs for row in out["tuning"]]),
        "mlp_history": pd.concat([h for out in outputs for h in out["history"]]),
        "confusion_matrices": metrics[["protocol", "seed", "model", "split", "threshold_rule",
                                       "threshold", "tn", "fp", "fn", "tp"]],
        "splits": runs["split_info"],
    }
    for name, table in tables.items():
        table.to_csv(results_dir / f"{name}.csv", index=False)
    predictions = pd.concat([p for out in outputs for p in out["predictions"].values()])
    for (protocol, model), part in predictions.groupby(["protocol", "model"]):
        part.to_csv(results_dir / "predictions" / f"{protocol}_{model}.csv", index=False)
    tables["predictions"] = predictions
    return tables


def summarize_random_seeds(metrics: pd.DataFrame) -> pd.DataFrame:
    """Mean and std over the random-split seeds (test split, tuned threshold)."""
    test = metrics[(metrics["protocol"] == "random") & (metrics["split"] == "test")
                   & (metrics["threshold_rule"] == "val_f1")]
    columns = ["accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc", "brier"]
    summary = test.groupby("model")[columns].agg(["mean", "std"])
    summary.columns = [f"{metric}_{stat}" for metric, stat in summary.columns]
    return summary.reset_index().assign(n_seeds=test["seed"].nunique())


def select_best_model(metrics: pd.DataFrame) -> str:
    """Best protocol-A model by validation PR-AUC (ROC-AUC as tiebreaker). Never uses test."""
    val = metrics[(metrics["protocol"] == "temporal") & (metrics["split"] == "val")
                  & (metrics["threshold_rule"] == "val_f1")]
    return val.sort_values(["pr_auc", "roc_auc"], ascending=False).iloc[0]["model"]


def save_selected_models(runs: dict, best: str, config: dict, results_dir: Path) -> dict:
    """Save the best model name, chosen hyperparameters and thresholds to JSON."""
    outputs = runs["outputs"].items()
    selected = {"best_model": best, "seed": config["seed"],
                "params": {f"{p}_seed{s}": out["params"] for (p, s), out in outputs},
                "thresholds": {f"{p}_seed{s}": out["thresholds"] for (p, s), out in outputs}}
    (Path(results_dir) / "selected_models.json").write_text(json.dumps(selected, indent=2))
    return selected

Run all protocols (A tuned; A2 reuses A's hyperparameters; B tuned separately for each of the 5 seeds).
This is the slowest cell (a few minutes on a laptop CPU).

In [ ]:
start = time.perf_counter()
runs = run_all_protocols(dataset, CONFIG)
tables = save_protocol_results(runs, RESULTS_DIR)
summarize_random_seeds(tables["metrics"]).to_csv(RESULTS_DIR / "random_seeds_summary.csv", index=False)
best_model = select_best_model(tables["metrics"])
save_selected_models(runs, best_model, CONFIG, RESULTS_DIR)
print(f"Training finished in {time.perf_counter() - start:.0f} s; best model on validation: {best_model}")

Every configuration that was tried for protocol A (selected rows marked):

In [ ]:
tuning = tables["tuning"]
tuning[tuning["protocol"] == "temporal"].round(4)

### MLP accuracy and loss curves (protocol A)

In [ ]:
def plot_mlp_history(results_dir: Path, style: str):
    """Fig. 8: MLP loss and accuracy per epoch (protocol A), with the early-stopping epoch."""
    history = pd.read_csv(Path(results_dir) / "mlp_history.csv")
    history = history[history["protocol"] == "temporal"]
    best = int(history["best_epoch"].iloc[0])
    fig, axes = plt.subplots(1, 2, figsize=figure_size(style))
    for ax, metric, title in [(axes[0], "loss", "Log loss (class-weighted)"),
                              (axes[1], "accuracy", "Accuracy (threshold 0.5)")]:
        ax.plot(history["epoch"], history[f"train_{metric}"], color="#1F5FA8", lw=2,
                label="Train (2011-2012)")
        ax.plot(history["epoch"], history[f"val_{metric}"], color="#A8822C", lw=2,
                label="Validation (2013-2014)")
        ax.axvline(best, color=MUTED, ls="--", lw=1.2,
                   label=f"Early stopping: best epoch {best}")
        ax.set(xlabel="Epoch", title=title)
    bottom_legend(fig, axes[0], ncol=3 if style == "slide" else 2,
                  height=0.1 if style == "slide" else 0.16)
    return fig

In [ ]:
fig = plot_mlp_history(RESULTS_DIR, "paper")
save_figure(fig, FIGURES_DIR, "fig08_mlp_curves", ["png", "pdf"])
plt.show()

### Cross-validation (strategy described in Section 7)

In [ ]:
def run_cv_all(runs: dict, config: dict, results_dir: Path) -> pd.DataFrame:
    """5-fold CV on protocol A's training cycle and on protocol B's train+validation."""
    settings = base_settings(config)
    rows = []
    for protocol, parts in [("temporal", ["train"]), ("random", ["train", "val"])]:
        out = runs["outputs"][(protocol, config["seed"])]
        dev = pd.concat([out["splits"][p] for p in parts])
        rows += cross_validate(protocol, dev, out["params"], out["best_epoch"], config, settings)
    folds = pd.DataFrame(rows)
    folds.to_csv(Path(results_dir) / "cv_folds.csv", index=False)
    summary = folds.groupby(["protocol", "model"]).agg(["mean", "std"]).drop(columns="fold")
    summary.columns = [f"{metric}_{stat}" for metric, stat in summary.columns]
    summary.reset_index().to_csv(Path(results_dir) / "cv_metrics.csv", index=False)
    return folds


def cv_summary_table(results_dir: Path) -> pd.DataFrame:
    """Per-fold ROC-AUC plus mean ± std for each protocol and model."""
    folds = pd.read_csv(Path(results_dir) / "cv_folds.csv")
    wide = folds.pivot_table(index=["protocol", "model"], columns="fold", values="roc_auc")
    wide.columns = [f"fold_{c}" for c in wide.columns]
    stats = folds.groupby(["protocol", "model"])["roc_auc"].agg(["mean", "std"])
    return wide.join(stats).reset_index()

In [ ]:
cv_folds = run_cv_all(runs, CONFIG, RESULTS_DIR)
display(pd.read_csv(RESULTS_DIR / "cv_metrics.csv").round(3))
cv_summary_table(RESULTS_DIR).round(3)

## 9. Model evaluation

For each model, protocol and split we compute accuracy, precision, recall, F1, ROC-AUC, PR-AUC, the
Brier score and the confusion matrix, at threshold 0.5 and at the threshold that maximises F1 on
**validation** data. Inference time per 1,000 rows and training time are recorded too.

### Uncertainty
95% bootstrap confidence intervals (1,000 resamples) are computed for test ROC-AUC and PR-AUC.
For two models on the same test set we use a **paired** bootstrap (both models scored on identical
resamples). We only call a difference real when its CI excludes zero.

In [ ]:
def bootstrap_indices(n: int, n_resamples: int, seed: int) -> np.ndarray:
    """Matrix of resampled row indices (one row per bootstrap resample)."""
    return np.random.default_rng(seed).integers(0, n, size=(n_resamples, n))


def bootstrap_metric(y, prob, metric, indices: np.ndarray) -> np.ndarray:
    """Value of `metric(y, prob)` on every bootstrap resample (NaN if one class is missing)."""
    y, prob = np.asarray(y), np.asarray(prob)
    values = np.full(len(indices), np.nan)
    for i, idx in enumerate(indices):
        if y[idx].min() != y[idx].max():
            values[i] = metric(y[idx], prob[idx])
    return values


def percentile_ci(values: np.ndarray, confidence: float) -> tuple[float, float]:
    """Percentile confidence interval, e.g. the 2.5th and 97.5th percentiles for 95%."""
    alpha = (1 - confidence) / 2
    return float(np.nanquantile(values, alpha)), float(np.nanquantile(values, 1 - alpha))


def metric_ci(y, prob, n_resamples: int, confidence: float, seed: int) -> dict:
    """Point estimates and bootstrap CIs for ROC-AUC and PR-AUC on one test set."""
    indices = bootstrap_indices(len(y), n_resamples, seed)
    result = {}
    for name, metric in [("roc_auc", roc_auc_score), ("pr_auc", average_precision_score)]:
        low, high = percentile_ci(bootstrap_metric(y, prob, metric, indices), confidence)
        result.update({name: metric(y, prob), f"{name}_low": low, f"{name}_high": high})
    return result


def paired_difference(y, prob_a, prob_b, metric, n_resamples: int, confidence: float,
                      seed: int) -> dict:
    """Paired bootstrap of metric(A) - metric(B) on the SAME test set.

    Both models are scored on identical resamples, so the CI reflects only the
    difference between models. If the CI contains 0 the difference is inconclusive.
    """
    indices = bootstrap_indices(len(y), n_resamples, seed)
    diffs = (bootstrap_metric(y, prob_a, metric, indices)
             - bootstrap_metric(y, prob_b, metric, indices))
    low, high = percentile_ci(diffs, confidence)
    return {"diff": metric(y, prob_a) - metric(y, prob_b), "diff_low": low, "diff_high": high}


def unpaired_difference(y_a, prob_a, y_b, prob_b, metric, n_resamples: int,
                        confidence: float, seed: int) -> dict:
    """Bootstrap of metric(set A) - metric(set B) for two DIFFERENT test sets.

    Each set is resampled independently (used e.g. for in-period vs future test).
    """
    values_a = bootstrap_metric(y_a, prob_a, metric, bootstrap_indices(len(y_a), n_resamples, seed))
    values_b = bootstrap_metric(y_b, prob_b, metric,
                                bootstrap_indices(len(y_b), n_resamples, seed + 1))
    low, high = percentile_ci(values_a - values_b, confidence)
    return {"diff": metric(y_a, prob_a) - metric(y_b, prob_b), "diff_low": low, "diff_high": high}


def ci_table(predictions: pd.DataFrame, n_resamples: int, confidence: float,
             seed: int) -> pd.DataFrame:
    """Bootstrap CIs of ROC-AUC and PR-AUC for every (protocol, seed, model, split) group.

    `predictions` has columns protocol, seed, model, split, label, probability.
    """
    rows = []
    for keys, group in predictions.groupby(["protocol", "seed", "model", "split"], sort=False):
        y, prob = group["label"].to_numpy(), group["probability"].to_numpy()
        rows.append({**dict(zip(["protocol", "seed", "model", "split"], keys)),
                     **metric_ci(y, prob, n_resamples, confidence, seed),
                     "n": len(y), "n_pos": int(y.sum())})
    return pd.DataFrame(rows)


def paired_comparisons(test: pd.DataFrame, reference: str, n_resamples: int,
                       confidence: float, seed: int) -> pd.DataFrame:
    """Paired bootstrap: the reference model minus every other model on one test set."""
    wide = test.pivot(index="SEQN", columns="model", values="probability")
    y = test.drop_duplicates("SEQN").set_index("SEQN").loc[wide.index, "label"].to_numpy()
    rows = []
    for other in [m for m in wide.columns if m != reference]:
        for name, metric in [("roc_auc", roc_auc_score), ("pr_auc", average_precision_score)]:
            result = paired_difference(y, wide[reference].to_numpy(), wide[other].to_numpy(),
                                       metric, n_resamples, confidence, seed)
            conclusive = result["diff_low"] > 0 or result["diff_high"] < 0
            rows.append({"reference": reference, "other": other, "metric": name, **result,
                         "ci_excludes_zero": conclusive})
    return pd.DataFrame(rows)


def gap_table(first: pd.DataFrame, second: pd.DataFrame, label: str, n_resamples: int,
              confidence: float, seed: int) -> pd.DataFrame:
    """Per model: metric(first test set) - metric(second test set) with a bootstrap CI.

    Used for "in-period holdout minus future cycle" and "random minus temporal".
    """
    rows = []
    for model in first["model"].unique():
        a, b = first[first["model"] == model], second[second["model"] == model]
        for name, metric in [("roc_auc", roc_auc_score), ("pr_auc", average_precision_score)]:
            result = unpaired_difference(a["label"].to_numpy(), a["probability"].to_numpy(),
                                         b["label"].to_numpy(), b["probability"].to_numpy(),
                                         metric, n_resamples, confidence, seed)
            rows.append({"comparison": label, "model": model, "metric": name, **result,
                         "ci_excludes_zero": result["diff_low"] > 0 or result["diff_high"] < 0})
    return pd.DataFrame(rows)


def run_uncertainty(predictions: pd.DataFrame, best_model: str, config: dict,
                    results_dir: Path) -> dict[str, pd.DataFrame]:
    """Bootstrap CIs, paired model comparisons and period-gap CIs; saved to CSV."""
    boot, seed = config["bootstrap"], config["seed"]
    n, conf = boot["n_resamples"], boot["confidence"]
    main = predictions[predictions["seed"] == seed]
    temporal_test = main[(main["protocol"] == "temporal") & (main["split"] == "test")]
    in_period = main[main["protocol"] == "in_period"]
    random_test = main[(main["protocol"] == "random") & (main["split"] == "test")]
    tables = {
        "metrics_ci": ci_table(main, n, conf, seed),
        "paired_comparisons": paired_comparisons(temporal_test, best_model, n, conf, seed),
        "period_gaps": pd.concat([
            gap_table(in_period[in_period["split"] == "test_in_period"],
                      in_period[in_period["split"] == "test"],
                      "A2: in-period holdout minus 2015-2016", n, conf, seed),
            gap_table(random_test, temporal_test, "B random test minus A temporal test",
                      n, conf, seed)]),
    }
    for name, table in tables.items():
        table.to_csv(Path(results_dir) / f"{name}.csv", index=False)
    return tables

In [ ]:
uncertainty = run_uncertainty(tables["predictions"], best_model, CONFIG, RESULTS_DIR)
uncertainty["paired_comparisons"].round(4)

### Model comparison table: chronological test set (2015-2016), validation-tuned threshold

In [ ]:
def model_comparison_table(results_dir: Path, protocol: str = "temporal",
                           split: str = "test") -> pd.DataFrame:
    """All metrics of every model on one test set (tuned threshold) with bootstrap CIs."""
    metrics = pd.read_csv(Path(results_dir) / "metrics.csv")
    ci = pd.read_csv(Path(results_dir) / "metrics_ci.csv")
    seed = int(metrics["seed"].min())
    rows = metrics[(metrics["protocol"] == protocol) & (metrics["split"] == split)
                   & (metrics["threshold_rule"] == "val_f1") & (metrics["seed"] == seed)]
    ci = ci[(ci["protocol"] == protocol) & (ci["split"] == split) & (ci["seed"] == seed)]
    table = rows.merge(ci[["model", "roc_auc_low", "roc_auc_high", "pr_auc_low", "pr_auc_high"]],
                       on="model")
    columns = ["model", "accuracy", "precision", "recall", "f1", "roc_auc", "roc_auc_low",
               "roc_auc_high", "pr_auc", "pr_auc_low", "pr_auc_high", "brier", "threshold",
               "train_time_s", "infer_ms_per_1k", "n", "n_pos"]
    table = table[columns].copy()
    table["model"] = table["model"].map(MODEL_LABELS)
    return table.reset_index(drop=True)

In [ ]:
model_comparison_table(RESULTS_DIR).round(3)

In [ ]:
def plot_auc_with_ci(results_dir: Path, style: str):
    """Fig. 3: temporal-test ROC-AUC and PR-AUC per model with 95% bootstrap CIs."""
    ci = pd.read_csv(Path(results_dir) / "metrics_ci.csv")
    ci = ci[(ci["protocol"] == "temporal") & (ci["split"] == "test")].set_index("model")
    order = models_in(ci.reset_index())
    fig, axes = plt.subplots(1, 2, figsize=figure_size(style))
    for ax, metric, title in [(axes[0], "roc_auc", "ROC-AUC"), (axes[1], "pr_auc", "PR-AUC")]:
        for y, model in enumerate(order):
            row = ci.loc[model]
            ax.errorbar(row[metric], y, xerr=[[row[metric] - row[f"{metric}_low"]],
                                              [row[f"{metric}_high"] - row[metric]]],
                        fmt="o", ms=8, capsize=4, color=MODEL_COLORS[model], lw=2)
            ax.annotate(f"{row[metric]:.3f}", (row[f"{metric}_high"], y), xytext=(6, 0),
                        textcoords="offset points", va="center", color=INK)
        ax.set_yticks(range(len(order)), [MODEL_LABELS[m] for m in order])
        ax.invert_yaxis()
        ax.set(title=f"{title} on 2015-2016 (95% CI)", xlabel=title)
        ax.margins(x=0.25)
    axes[1].set_yticklabels([])
    fig.tight_layout()
    return fig


def plot_roc_curves(results_dir: Path, style: str):
    """Fig. 5: ROC curves of all models on the chronological test cycle (2015-2016)."""
    fig, ax = plt.subplots(figsize=figure_size(style, 0.62 if style == "paper" else 0.5))
    for model, label in MODEL_LABELS.items():
        path = Path(results_dir) / "predictions" / f"temporal_{model}.csv"
        pred = pd.read_csv(path)
        pred = pred[pred["split"] == "test"]
        fpr, tpr, _ = roc_curve(pred["label"], pred["probability"])
        ax.plot(fpr, tpr, color=MODEL_COLORS[model], lw=2, label=label)
    ax.plot([0, 1], [0, 1], ls="--", color=MUTED, lw=1, label="Chance")
    ax.set(xlabel="False positive rate", ylabel="True positive rate",
           title="ROC curves, chronological test set (2015-2016)", aspect="equal")
    ax.legend(loc="lower right")
    fig.tight_layout()
    return fig


def plot_confusion_matrix(results_dir: Path, style: str):
    """Fig. 6: confusion matrix of the validation-selected model on the 2015-2016 test set."""
    best = json.loads((Path(results_dir) / "selected_models.json").read_text())["best_model"]
    cm = pd.read_csv(Path(results_dir) / "confusion_matrices.csv")
    row = cm[(cm["protocol"] == "temporal") & (cm["model"] == best) & (cm["split"] == "test")
             & (cm["threshold_rule"] == "val_f1")].iloc[0]
    matrix = np.array([[row["tn"], row["fp"]], [row["fn"], row["tp"]]])
    fig, ax = plt.subplots(figsize=(6.5, 5.2) if style == "slide" else (4.2, 3.6))
    ax.imshow(matrix, cmap="Blues", vmin=0)
    ax.grid(False)
    for (i, j), value in np.ndenumerate(matrix):
        ax.text(j, i, f"{value:,}\n({value / matrix.sum():.1%})", ha="center", va="center",
                color="white" if value > matrix.max() / 2 else INK)
    ax.set_xticks([0, 1], ["No diabetes", "Diabetes"])
    ax.set_yticks([0, 1], ["No diabetes", "Diabetes"])
    ax.set(xlabel="Predicted", ylabel="Actual (self-reported)",
           title=f"{MODEL_LABELS[best]}, 2015-2016 test\n(threshold {row['threshold']:.2f})")
    fig.tight_layout()
    return fig

In [ ]:
for name, plot in [("fig03_auc_ci", plot_auc_with_ci), ("fig05_roc_curves", plot_roc_curves),
                   ("fig06_confusion_matrix", plot_confusion_matrix)]:
    fig = plot(RESULTS_DIR, "paper")
    save_figure(fig, FIGURES_DIR, name, ["png", "pdf"])
    plt.show()

Confusion matrices of all models on the chronological test set:

In [ ]:
cm = pd.read_csv(RESULTS_DIR / "confusion_matrices.csv")
cm[(cm["protocol"] == "temporal") & (cm["split"] == "test")].round(3)

### Ablation study
The model selected on validation in protocol A is retrained with the same split, seed and hyperparameters,
changing one thing at a time. Scores are on the 2015-2016 test set.

In [ ]:
def ablation_variants(config: dict, settings: dict) -> list[tuple[str, dict, bool]]:
    """(name, settings, use_tuned_threshold) for every ablation configuration."""
    variants = [("full model", settings, True)]
    for group, columns in config["data"]["feature_groups"].items():
        reduced = {**settings,
                   "numeric": [c for c in settings["numeric"] if c not in columns],
                   "categorical": [c for c in settings["categorical"] if c not in columns]}
        variants.append((f"without {group} features", reduced, True))
    variants.append(("no class-imbalance handling", {**settings, "balanced": False}, True))
    variants.append(("threshold 0.5 instead of tuned", settings, False))
    return variants


def run_ablation(name: str, params: dict, splits: dict, config: dict,
                 settings: dict) -> pd.DataFrame:
    """Retrain the selected model with one component changed at a time.

    Same split, seed and hyperparameters as the full model; scores are on the
    temporal test set and deltas are relative to the full model.
    """
    rows = []
    for variant, variant_settings, tuned in ablation_variants(config, settings):
        fitted = fit_on_splits(name, params, splits, variant_settings)
        metrics, _, _ = evaluate_model(name, fitted, splits, variant_settings, {})
        rule = "val_f1" if tuned else "0.5"
        test = next(m for m in metrics if m["split"] == "test" and m["threshold_rule"] == rule)
        rows.append({"model": name, "configuration": variant, "threshold": test["threshold"],
                     "roc_auc": test["roc_auc"], "pr_auc": test["pr_auc"], "f1": test["f1"],
                     "accuracy": test["accuracy"]})
    ablation = pd.DataFrame(rows)
    for metric in ["roc_auc", "pr_auc", "f1"]:
        ablation[f"delta_{metric}"] = ablation[metric] - ablation.loc[0, metric]
    return ablation


def plot_ablation(results_dir: Path, style: str):
    """Fig. 9: change in test ROC-AUC, PR-AUC and F1 when one component is removed."""
    ablation = pd.read_csv(Path(results_dir) / "ablation.csv").iloc[1:]  # row 0 = full model
    fig, ax = plt.subplots(figsize=figure_size(style, 0.5))
    y = np.arange(len(ablation))
    for i, (metric, color) in enumerate([("roc_auc", "#1F5FA8"), ("pr_auc", "#A8822C"),
                                         ("f1", "#109A92")]):
        ax.barh(y + (i - 1) * 0.26, ablation[f"delta_{metric}"], 0.24, color=color,
                label=f"Δ {metric.replace('_', '-').upper()}")
    ax.axvline(0, color=MUTED, lw=1)
    ax.set_yticks(y, ablation["configuration"])
    ax.invert_yaxis()
    model = MODEL_LABELS[ablation["model"].iloc[0]]
    ax.set(xlabel="Change vs full model on 2015-2016 test set", title=f"Ablation study ({model})")
    ax.legend(loc="lower left")
    fig.tight_layout()
    return fig

In [ ]:
temporal_run = runs["outputs"][("temporal", CONFIG["seed"])]
ablation = run_ablation(best_model, temporal_run["params"][best_model], temporal_run["splits"],
                        CONFIG, base_settings(CONFIG))
ablation.to_csv(RESULTS_DIR / "ablation.csv", index=False)
display(ablation.round(4))
fig = plot_ablation(RESULTS_DIR, "paper")
save_figure(fig, FIGURES_DIR, "fig09_ablation", ["png", "pdf"])
plt.show()

## 10. Temporal drift analysis

We compare the 2013-2014 and 2015-2016 cycles with 2011-2012:
* **Numeric features:** means, medians, standard deviations and the two-sample **Kolmogorov-Smirnov (KS)
  statistic** (largest gap between the two cumulative distributions). With thousands of rows almost any
  difference has a tiny p-value, so we read the KS statistic itself as an effect size.
* **PSI** (Population Stability Index) with quantile bins fixed on 2011-2012.
* **Categorical features:** category shares and their change in percentage points.
* **Domain classifier:** can a logistic regression tell two cycles apart from the features alone?
  Cross-validated ROC-AUC ≈ 0.5 means little covariate shift.

These statistics describe **what** changed. They do not tell us **why** (for example, sampling design,
population change or diagnosis practice); we do not claim causes without external evidence.

In [ ]:
def psi(reference: np.ndarray, current: np.ndarray, n_bins: int) -> float:
    """Population Stability Index with quantile bins fixed on the reference cycle.

    PSI = sum over bins of (p_cur - p_ref) * ln(p_cur / p_ref). A common rule of
    thumb: < 0.1 small change, 0.1-0.25 moderate, > 0.25 large.
    """
    edges = np.unique(np.quantile(reference, np.linspace(0, 1, n_bins + 1)))
    edges[0], edges[-1] = -np.inf, np.inf
    p_ref = np.histogram(reference, edges)[0] / len(reference)
    p_cur = np.histogram(current, edges)[0] / len(current)
    p_ref, p_cur = np.clip(p_ref, 1e-6, None), np.clip(p_cur, 1e-6, None)  # avoid log(0)
    return float(np.sum((p_cur - p_ref) * np.log(p_cur / p_ref)))


def numeric_drift(dataset: pd.DataFrame, features: list[str], reference: str,
                  n_bins: int) -> pd.DataFrame:
    """Summary statistics, KS statistic and PSI of each numeric feature vs the reference.

    The KS statistic is the largest gap between the two cumulative distributions
    (0 = identical, 1 = no overlap). With thousands of rows almost any difference
    gives a tiny p-value, so the statistic itself is read as an effect size.
    """
    ref = dataset[dataset["cycle"] == reference]
    rows = []
    for cycle in sorted(c for c in dataset["cycle"].unique() if c != reference):
        cur = dataset[dataset["cycle"] == cycle]
        for feature in features:
            a, b = ref[feature].dropna().to_numpy(), cur[feature].dropna().to_numpy()
            test = ks_2samp(a, b)
            rows.append({"feature": feature, "reference": reference, "compared": cycle,
                         "mean_ref": a.mean(), "mean_cmp": b.mean(),
                         "median_ref": np.median(a), "median_cmp": np.median(b),
                         "std_ref": a.std(), "std_cmp": b.std(),
                         "missing_pct_ref": ref[feature].isna().mean() * 100,
                         "missing_pct_cmp": cur[feature].isna().mean() * 100,
                         "ks_statistic": test.statistic, "ks_pvalue": test.pvalue,
                         "psi": psi(a, b, n_bins)})
    return pd.DataFrame(rows)


def categorical_drift(dataset: pd.DataFrame, features: list[str], reference: str,
                      labels: dict) -> pd.DataFrame:
    """Share of each category per cycle and its change (percentage points) vs the reference."""
    rows = []
    for feature in features:
        shares = (dataset[feature].astype("Float64").fillna(-1)  # -1 marks missing values
                  .groupby(dataset["cycle"]).value_counts(normalize=True)
                  .unstack(level=0, fill_value=0.0) * 100)
        for code, row in shares.iterrows():
            name = "missing" if code == -1 else labels.get(feature, {}).get(int(code), str(code))
            for cycle in [c for c in shares.columns if c != reference]:
                rows.append({"feature": feature, "code": int(code), "category": name,
                             "reference": reference, "compared": cycle,
                             "pct_ref": row[reference], "pct_cmp": row[cycle],
                             "diff_pp": row[cycle] - row[reference]})
    return pd.DataFrame(rows)


def domain_classifier_auc(dataset: pd.DataFrame, reference: str, compared: str,
                          config: dict) -> dict:
    """Can a model tell the two cycles apart from the features alone?

    A logistic regression is trained to predict "is this row from `compared`?".
    Cross-validated ROC-AUC near 0.5 means the feature distributions are very
    similar (little covariate shift); values near 1 mean strong shift.
    """
    data_cfg = config["data"]
    features = data_cfg["numeric_features"] + data_cfg["categorical_features"]
    both = dataset[dataset["cycle"].isin([reference, compared])]
    X, y = feature_matrix(both, features), (both["cycle"] == compared).astype(int)
    model = Pipeline([("prep", make_preprocessor(data_cfg["numeric_features"],
                                                 data_cfg["categorical_features"])),
                      ("model", LogisticRegression(max_iter=2000))])
    folds = StratifiedKFold(n_splits=config["cv"]["n_splits"], shuffle=True,
                            random_state=config["seed"])
    scores = cross_val_score(model, X, y, cv=folds, scoring="roc_auc")
    return {"reference": reference, "compared": compared, "roc_auc_mean": scores.mean(),
            "roc_auc_std": scores.std(), "n_rows": len(both)}


def run_drift(dataset: pd.DataFrame, config: dict, results_dir) -> dict[str, pd.DataFrame]:
    """Compute every drift table and save them as CSV files in `results_dir`."""
    data_cfg, reference = config["data"], config["drift"]["reference_cycle"]
    others = sorted(c for c in dataset["cycle"].unique() if c != reference)
    tables = {
        "cycle_summary": cycle_summary(dataset),
        "drift_numeric": numeric_drift(dataset, data_cfg["numeric_features"], reference,
                                       config["drift"]["psi_bins"]),
        "drift_categorical": categorical_drift(dataset, data_cfg["categorical_features"],
                                               reference, data_cfg["category_labels"]),
        "domain_classifier": pd.DataFrame([domain_classifier_auc(dataset, reference, c, config)
                                           for c in others]),
    }
    for name, table in tables.items():
        table.to_csv(f"{results_dir}/{name}.csv", index=False)
    return tables


def plot_feature_drift(results_dir: Path, style: str):
    """Fig. 7: KS statistic of numeric features and category-share changes vs 2011-2012."""
    numeric = pd.read_csv(Path(results_dir) / "drift_numeric.csv")
    categorical = pd.read_csv(Path(results_dir) / "drift_categorical.csv")
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=figure_size(style, 0.5))
    features = list(dict.fromkeys(numeric["feature"]))
    for i, (cycle, part) in enumerate(numeric.groupby("compared")):
        ax1.barh(np.arange(len(features)) + i * 0.4, part.set_index("feature").loc[features,
                 "ks_statistic"], 0.38, color=CYCLE_COLORS[i + 1], label=f"{cycle} vs 2011-2012")
    ax1.set_yticks(np.arange(len(features)) + 0.2, features)
    ax1.invert_yaxis()
    ax1.set(xlabel="KS statistic (0 = identical)", title="Numeric features")
    last = categorical[(categorical["compared"] == categorical["compared"].max())
                       & (categorical["category"] != "missing")]
    last = last.reindex(last["diff_pp"].abs().sort_values(ascending=False).index).head(8)
    colors = ["#1F5FA8" if d > 0 else "#B4432E" for d in last["diff_pp"]]
    ax2.barh(last["feature"] + ": " + last["category"], last["diff_pp"], color=colors)
    ax2.axvline(0, color=MUTED, lw=1)
    ax2.invert_yaxis()
    ax2.set(xlabel="Change in share (percentage points)",
            title=f"Largest category shifts, {last['compared'].iloc[0]} vs 2011-2012")
    bottom_legend(fig, ax1, ncol=2, height=0.1)
    return fig

In [ ]:
drift = run_drift(dataset, CONFIG, RESULTS_DIR)
display(drift["drift_numeric"].round(4))
display(drift["domain_classifier"].round(4))
categorical = drift["drift_categorical"]
categorical[categorical["compared"] == "2015-2016"].round(2)

In [ ]:
fig = plot_feature_drift(RESULTS_DIR, "paper")
save_figure(fig, FIGURES_DIR, "fig07_feature_drift_ks", ["png", "pdf"])
plt.show()

## 11. Results and comparison

### Random vs in-period vs chronological evaluation
Mean ± std over the 5 random-split seeds (test set, tuned threshold):

In [ ]:
pd.read_csv(RESULTS_DIR / "random_seeds_summary.csv").round(3)

Differences between test sets with 95% bootstrap CIs (A2: the same fitted model on the in-period holdout
minus the 2015-2016 cycle; B minus A: random-split test minus chronological test):

In [ ]:
uncertainty["period_gaps"].round(4)

In [ ]:
def split_scores(metrics: pd.DataFrame, protocol: str, split: str) -> pd.DataFrame:
    """Rows of one protocol and split, indexed by model."""
    return metrics[(metrics["protocol"] == protocol) & (metrics["split"] == split)].set_index("model")


def plot_protocol_comparison(results_dir: Path, style: str):
    """Fig. 4: test ROC-AUC under the random split, the in-period holdout and the future cycle."""
    metrics = pd.read_csv(Path(results_dir) / "metrics.csv")
    rule = metrics[(metrics["threshold_rule"] == "val_f1")]
    random = (rule[(rule["protocol"] == "random") & (rule["split"] == "test")]
              .groupby("model")["roc_auc"].agg(["mean", "std"]))
    n_seeds = rule.loc[rule["protocol"] == "random", "seed"].nunique()
    series = [(f"B: random split (mean ± std, {n_seeds} seeds)", random["mean"], random["std"],
               "#9FB0CC", "s"),
              ("A2: in-period holdout (2011-2012)",
               split_scores(rule, "in_period", "test_in_period")["roc_auc"], None, "#4F6A99", "^"),
              ("A: chronological test (2015-2016)", split_scores(rule, "temporal", "test")["roc_auc"],
               None, "#14213D", "o")]
    order = models_in(rule)
    fig, ax = plt.subplots(figsize=figure_size(style, 0.5))
    for i, (label, values, std, color, marker) in enumerate(series):
        y = np.arange(len(order)) + (i - 1) * 0.22
        ax.errorbar(values[order], y, xerr=None if std is None else std[order], fmt=marker,
                    ms=8, capsize=3, color=color, label=label, lw=1.5)
    ax.set_yticks(range(len(order)), [MODEL_LABELS[m] for m in order])
    ax.invert_yaxis()
    ax.set(xlabel="Test ROC-AUC", title="Random vs in-period vs chronological evaluation")
    bottom_legend(fig, ax, ncol=3 if style == "slide" else 2,
                  height=0.12 if style == "slide" else 0.15)
    return fig

In [ ]:
fig = plot_protocol_comparison(RESULTS_DIR, "paper")
save_figure(fig, FIGURES_DIR, "fig04_random_vs_temporal", ["png", "pdf"])
plt.show()

### Key findings (computed from the saved result files)

In [ ]:
def lookup(results_dir: Path, source: str, column: str, row_filter: dict | None = None,
           agg: str = "single"):
    """Read one value from a result file.

    `source` is a CSV (filtered by `row_filter`, then reduced with agg = single,
    sum or max) or a JSON file (then `column` is a dot-separated key path).
    """
    path = Path(results_dir) / source
    if source.endswith(".json"):
        value = json.loads(path.read_text())
        for key in column.split("."):
            value = value[key]
        return value
    table = pd.read_csv(path)
    for key, wanted in (row_filter or {}).items():
        table = table[table[key] == wanted]
    if agg == "sum":
        return float(table[column].sum())
    if agg == "max":
        return float(table[column].max())
    if len(table) != 1:
        raise ValueError(f"{source} with filter {row_filter} matched {len(table)} rows, not 1")
    value = table[column].iloc[0]
    return value.item() if hasattr(value, "item") else value


def key_number_specs(results_dir: Path) -> list[dict]:
    """Definitions (source file, column, filter) of the headline numbers."""
    best = lookup(results_dir, "selected_models.json", "best_model")
    seed = lookup(results_dir, "selected_models.json", "seed")
    test = {"protocol": "temporal", "split": "test", "model": best}
    specs = [{"id": "best_model", "label": "Model selected on 2013-2014 validation PR-AUC",
              "source": "selected_models.json", "column": "best_model"},
             {"id": "seed", "label": "Main random seed", "source": "selected_models.json",
              "column": "seed"}]
    for cycle in ["2011-2012", "2013-2014", "2015-2016"]:
        tag = cycle[:4]
        specs += [{"id": f"n_{tag}", "label": f"Eligible adults {cycle}", "source":
                   "cycle_summary.csv", "column": "n", "filter": {"cycle": cycle}},
                  {"id": f"prevalence_{tag}", "label": f"Unweighted prevalence {cycle}",
                   "source": "cycle_summary.csv", "column": "prevalence",
                   "filter": {"cycle": cycle}},
                  {"id": f"weighted_prevalence_{tag}", "label": f"Weighted prevalence {cycle}",
                   "source": "cycle_summary.csv", "column": "weighted_prevalence",
                   "filter": {"cycle": cycle}}]
    specs.append({"id": "n_total", "label": "Eligible adults, all cycles",
                  "source": "cycle_summary.csv", "column": "n", "agg": "sum"})
    for column in ["roc_auc", "roc_auc_low", "roc_auc_high", "pr_auc", "pr_auc_low",
                   "pr_auc_high"]:
        specs.append({"id": f"best_temporal_{column}", "label": f"Best model, 2015-2016 {column}",
                      "source": "metrics_ci.csv", "column": column, "filter": test})
    for column in ["f1", "recall", "precision", "accuracy", "threshold"]:
        specs.append({"id": f"best_temporal_{column}", "label": f"Best model, 2015-2016 {column}"
                      " (validation-tuned threshold)", "source": "metrics.csv", "column": column,
                      "filter": {**test, "threshold_rule": "val_f1", "seed": seed}})
    gap_a2 = {"comparison": "A2: in-period holdout minus 2015-2016", "model": best,
              "metric": "roc_auc"}
    gap_b = {"comparison": "B random test minus A temporal test", "model": best,
             "metric": "roc_auc"}
    for name, row_filter in [("a2_gap", gap_a2), ("random_minus_temporal", gap_b)]:
        for column in ["diff", "diff_low", "diff_high"]:
            specs.append({"id": f"{name}_roc_auc_{column}", "label": f"{name} ROC-AUC {column}",
                          "source": "period_gaps.csv", "column": column, "filter": row_filter})
    specs += [
        {"id": "random_roc_auc_mean", "label": "Best model ROC-AUC, random split (mean of seeds)",
         "source": "random_seeds_summary.csv", "column": "roc_auc_mean", "filter": {"model": best}},
        {"id": "random_n_seeds", "label": "Number of random-split seeds",
         "source": "random_seeds_summary.csv", "column": "n_seeds", "filter": {"model": best}},
        {"id": "random_roc_auc_std", "label": "Best model ROC-AUC, random split (std of seeds)",
         "source": "random_seeds_summary.csv", "column": "roc_auc_std", "filter": {"model": best}},
        {"id": "cv_temporal_roc_auc_mean", "label": "Best model 5-fold CV ROC-AUC (2011-2012)",
         "source": "cv_metrics.csv", "column": "roc_auc_mean",
         "filter": {"protocol": "temporal", "model": best}},
        {"id": "cv_temporal_roc_auc_std", "label": "Best model 5-fold CV ROC-AUC std",
         "source": "cv_metrics.csv", "column": "roc_auc_std",
         "filter": {"protocol": "temporal", "model": best}},
        {"id": "domain_auc_2015", "label": "Domain classifier ROC-AUC, 2011-2012 vs 2015-2016",
         "source": "domain_classifier.csv", "column": "roc_auc_mean",
         "filter": {"compared": "2015-2016"}},
        {"id": "max_ks_2015", "label": "Largest KS statistic, 2015-2016 vs 2011-2012",
         "source": "drift_numeric.csv", "column": "ks_statistic",
         "filter": {"compared": "2015-2016"}, "agg": "max"},
        {"id": "ablation_demographic_delta_roc_auc", "label": "ROC-AUC change without demographics",
         "source": "ablation.csv", "column": "delta_roc_auc",
         "filter": {"configuration": "without demographic features"}},
        {"id": "ablation_anthropometric_delta_roc_auc",
         "label": "ROC-AUC change without body measurements", "source": "ablation.csv",
         "column": "delta_roc_auc", "filter": {"configuration": "without anthropometric features"}},
        {"id": "mlp_best_epoch", "label": "MLP early-stopping epoch (protocol A)",
         "source": "mlp_history.csv", "column": "best_epoch",
         "filter": {"protocol": "temporal"}, "agg": "max"},
    ]
    return specs


def key_numbers(results_dir: Path) -> list[dict]:
    """Evaluate every key-number spec; each entry keeps its source for traceability."""
    entries = []
    for spec in key_number_specs(results_dir):
        value = lookup(results_dir, spec["source"], spec["column"], spec.get("filter"),
                       spec.get("agg", "single"))
        entries.append({**spec, "value": value})
    return entries


def ci_text(numbers: dict, prefix: str) -> str:
    """Format the 95% CI stored under `<prefix>_low` / `<prefix>_high` as "[low, high]"."""
    return f"[{numbers[prefix + '_low']:.3f}, {numbers[prefix + '_high']:.3f}]"


def key_findings(results_dir: Path) -> str:
    """Markdown bullet list of the main findings, with every number read from the results."""
    k = {e["id"]: e["value"] for e in key_numbers(results_dir)}
    name = MODEL_LABELS[k["best_model"]]
    lines = [
        (f"- Eligible adults: {k['n_total']:,.0f}; unweighted prevalence "
         f"{k['prevalence_2011']:.1%} (2011-2012), {k['prevalence_2013']:.1%} (2013-2014), "
         f"{k['prevalence_2015']:.1%} (2015-2016)."),
        (f"- Model selected on validation: **{name}**. Chronological test (2015-2016) ROC-AUC "
         f"{k['best_temporal_roc_auc']:.3f} {ci_text(k, 'best_temporal_roc_auc')}, PR-AUC "
         f"{k['best_temporal_pr_auc']:.3f} {ci_text(k, 'best_temporal_pr_auc')}, F1 "
         f"{k['best_temporal_f1']:.3f} at threshold {k['best_temporal_threshold']:.2f}."),
        (f"- Same model, random split: ROC-AUC {k['random_roc_auc_mean']:.3f} ± "
         f"{k['random_roc_auc_std']:.3f} ({k['random_n_seeds']} seeds); seed-{k['seed']} random split "
         f"minus chronological test {k['random_minus_temporal_roc_auc_diff']:+.3f} "
         f"{ci_text(k, 'random_minus_temporal_roc_auc_diff')}."),
        (f"- Protocol A2 (same fitted model): in-period holdout minus 2015-2016 ROC-AUC "
         f"{k['a2_gap_roc_auc_diff']:+.3f} {ci_text(k, 'a2_gap_roc_auc_diff')}."),
        (f"- Covariate shift 2011-2012 vs 2015-2016: largest numeric KS statistic "
         f"{k['max_ks_2015']:.3f}; domain-classifier ROC-AUC {k['domain_auc_2015']:.3f}."),
        (f"- Ablation: dropping demographic features changes ROC-AUC by "
         f"{k['ablation_demographic_delta_roc_auc']:+.3f}; dropping body measurements by "
         f"{k['ablation_anthropometric_delta_roc_auc']:+.3f}."),
    ]
    return "\n".join(lines)


def describe_interval(low: float, high: float) -> str:
    """Plain-language reading of a confidence interval for a difference."""
    if low > 0:
        return "positive (the 95% CI excludes zero)"
    if high < 0:
        return "negative (the 95% CI excludes zero)"
    return "inconclusive (the 95% CI includes zero)"


def interpretation(results_dir: Path) -> str:
    """Rule-based reading of the uncertainty results, so wording always matches the numbers."""
    paired = pd.read_csv(Path(results_dir) / "paired_comparisons.csv")
    roc = paired[paired["metric"] == "roc_auc"]
    k = {e["id"]: e["value"] for e in key_numbers(results_dir)}
    best = MODEL_LABELS[k["best_model"]]
    clear = roc[roc["ci_excludes_zero"]]
    beaten = ", ".join(MODEL_LABELS[m] for m in clear[clear["diff"] > 0]["other"]) or "none"
    worse = ", ".join(MODEL_LABELS[m] for m in clear[clear["diff"] < 0]["other"]) or "none"
    lines = [
        (f"- Paired bootstrap on 2015-2016: {best} has a clearly higher ROC-AUC than: {beaten}; "
         f"clearly lower than: {worse}; all other pairwise differences are inconclusive."),
        ("- Random-split minus chronological ROC-AUC (selected model) is "
         + describe_interval(k["random_minus_temporal_roc_auc_diff_low"],
                             k["random_minus_temporal_roc_auc_diff_high"]) + "."),
        ("- In-period holdout minus future-cycle ROC-AUC (protocol A2, selected model) is "
         + describe_interval(k["a2_gap_roc_auc_diff_low"], k["a2_gap_roc_auc_diff_high"]) + "."),
    ]
    return "\n".join(lines)

In [ ]:
display(Markdown(key_findings(RESULTS_DIR)))
display(Markdown(interpretation(RESULTS_DIR)))

### Reproducibility record
The metrics of this run are saved to `./notebook_outputs/metrics.csv`, together with package versions
and hardware details.

In [ ]:
def package_versions(packages: list[str]) -> dict[str, str]:
    """Installed version of each package (or 'not installed')."""
    versions = {}
    for package in packages:
        try:
            versions[package] = metadata.version(package)
        except metadata.PackageNotFoundError:
            versions[package] = "not installed"
    return versions


def hardware_info() -> dict:
    """CPU model, core count, RAM and operating system of this machine."""
    info = {"os": platform.platform(), "machine": platform.machine(),
            "cpu_count": os.cpu_count(), "cpu_model": platform.processor()}
    try:
        if sys.platform == "darwin":
            info["cpu_model"] = subprocess.run(["sysctl", "-n", "machdep.cpu.brand_string"],
                                               capture_output=True, text=True, check=False).stdout.strip()
            ram = int(subprocess.run(["sysctl", "-n", "hw.memsize"], capture_output=True, check=False,
                                     text=True).stdout)
        else:
            ram = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES")
        info["ram_gb"] = round(ram / 1024**3, 1)
    except (OSError, ValueError) as error:
        logger.warning("Could not read hardware details: %s", error)
    return info


def write_run_metadata(config: dict, results_dir: Path, extra: dict) -> dict:
    """Save Python/package versions, seed, timestamp and hardware to run_metadata.json.

    Stage runtimes from earlier runs are kept, so running stages one by one still
    produces a complete record.
    """
    path = Path(results_dir, "run_metadata.json")
    previous = json.loads(path.read_text()) if path.exists() else {}
    runtimes = {**previous.get("stage_runtimes_s", {}), **extra.get("stage_runtimes_s", {})}
    packages = ["pandas", "numpy", "scikit-learn", "xgboost", "lightgbm", "scipy",
                "matplotlib", "pyyaml", "requests"]
    meta = {"timestamp_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
            "python": platform.python_version(), "seed": config["seed"],
            "n_jobs": config["n_jobs"], "packages": package_versions(packages),
            "hardware": hardware_info(), **extra, "stage_runtimes_s": runtimes}
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(meta, indent=2))
    return meta

In [ ]:
metadata = write_run_metadata(CONFIG, RESULTS_DIR, {"source": "standalone notebook", "run_mode": RUN_MODE})
print(json.dumps(metadata, indent=2))
print(sorted(p.name for p in RESULTS_DIR.glob("*.csv")))

## 12. Limitations and conclusion

### Limitations
* **Survey, not hospital data:** NHANES is a cross-sectional population survey, not longitudinal
  electronic health records; each cycle contains different people.
* **Unweighted models:** NHANES uses complex multistage sampling with survey weights. Our models are
  trained without weights, so the results describe the *sample*, not the US population (weighted
  prevalence is reported only as a descriptive check).
* **Self-reported target:** the label is self-reported diagnosed diabetes, so undiagnosed diabetes counts
  as negative.
* **Reverse causation:** body measurements are taken after a possible diagnosis, so associations may
  partly reflect treatment or lifestyle changes after diagnosis.
* **Small scale:** three cycles, nine features and small hyperparameter grids; this is a prototype, not
  a full reproduction of TabReD.
* **Random vs temporal test sets differ** in who is in them, so their difference is not purely a time effect.

### Conclusion
The key findings and their rule-based interpretation in Section 11 summarise the measured results.
Students: write the final interpretation here in your own words, using only the numbers shown above.